# Bangla Fake News Detection under Class Imbalance — Data Mining Extension
**Classical classifiers, imbalance-mitigation strategies and BanglaBERT on the BanFakeNews dataset**

Author: Rapidul Islam (Data Mining course, solo paper) · Extends the Machine Learning course paper
*"A Comparative Analysis of Machine Learning Classifiers for Bangla Fake News Detection with Class Imbalance Mitigation"*.

This notebook is designed so that **every number and figure in the paper can be traced back to a file it writes**.
It follows the *Professional Writing Guide – Data Mining* rules:

| Guide rule | Where it is enforced in this notebook |
|---|---|
| Split → fit vectorizer on **train only** → transform test → resample **train only** | §6 `build_pipeline` (TF‑IDF *inside* an imblearn `Pipeline`, so it is re‑fit in every CV fold) |
| Tune hyper‑parameters with CV **inside the training data**; evaluate the final model **once** on the untouched test set | §8 (`GridSearchCV` on the training split only; test set is scored once per configuration) |
| Report per‑class P/R/F1, macro/weighted F1, confusion matrix, PR‑AUC, ROC‑AUC, CIs | §5 metrics, §10 tables/figures, bootstrap CIs |
| Compare imbalance strategies: none, class weights, random over/under‑sampling, SMOTE, a text‑appropriate alternative, threshold tuning | §6–§8 (6 conditions) + threshold‑tuned results |
| Duplicates / conflicting labels / missing text documented | §3 data audit |
| Seeds, model settings, environment, intermediate and failed results saved | §0–§2 manifest, `cache/`, `logs/` |

**Quick start:** set `PROFILE = "smoke"` in §1 and *Run all* (≈10 min) to check that everything works, then switch to
`"standard"` or `"full"`. All results are written to one output folder (Google Drive if enabled) and zipped at the end.
See `HOW_TO_RUN.md` for the full instructions.

## 0. Setup

In [ ]:
import os, sys, subprocess, importlib.util, warnings
os.environ["PYTHONWARNINGS"] = "ignore"          # also silences joblib worker processes
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
warnings.filterwarnings("ignore")

def pip_install(*pkgs):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print("pip install failed for", pkgs, "\n", r.stderr[-400:])
    return r.returncode == 0

_needed = {"imblearn": "imbalanced-learn", "lightgbm": "lightgbm", "transformers": "transformers"}
_missing = [pip for mod, pip in _needed.items() if importlib.util.find_spec(mod) is None]
if _missing:
    print("Installing:", _missing); pip_install(*_missing)

# Official BanglaBERT text normalizer (the model was pre-trained on normalized text)
if importlib.util.find_spec("normalizer") is None:
    ok = pip_install("git+https://github.com/csebuetnlp/normalizer")
    print("csebuetnlp/normalizer installed:", ok)
print("setup done")

In [ ]:
import json, math, time, re, glob, hashlib, shutil, platform, random, datetime as dt, copy, itertools, pickle
from pathlib import Path
from collections import OrderedDict

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from scipy import stats

import sklearn
from sklearn.base import BaseEstimator, clone
from sklearn.model_selection import (train_test_split, StratifiedKFold, RepeatedStratifiedKFold,
                                     StratifiedGroupKFold, GridSearchCV, cross_val_predict)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, StackingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import (roc_auc_score, average_precision_score, precision_recall_curve, roc_curve,
                             confusion_matrix)
try:
    import imblearn
    from imblearn.pipeline import Pipeline as ImbPipeline
    from imblearn.over_sampling import RandomOverSampler, SMOTE
    from imblearn.under_sampling import RandomUnderSampler
except Exception as _e:       # an old imbalanced-learn preinstalled next to a very new scikit-learn can fail to import
    pip_install("-U", "imbalanced-learn")
    raise RuntimeError("imbalanced-learn was upgraded because it could not be imported (" + repr(_e) +
                       "). Please use  Runtime > Restart session  and run the notebook again from the top.")
import lightgbm as lgb
from joblib import Parallel, delayed

IN_COLAB = "google.colab" in sys.modules
print(f"Python {platform.python_version()} | sklearn {sklearn.__version__} | imblearn {imblearn.__version__} | "
      f"lightgbm {lgb.__version__} | pandas {pd.__version__} | numpy {np.__version__}")

## 1. Configuration  ← the only cell you normally edit

* `PROFILE` controls how much computation is done (see table in `HOW_TO_RUN.md`).
* Everything else is derived from the profile; individual switches can be overridden in the *override* block.

In [ ]:
# ----------------------------------------------------------------------------------------------
PROFILE            = os.environ.get("BFN_PROFILE", "smoke")      # START WITH "smoke" (sanity check, ~10 min), then "standard" (~1.5-2.5 h) or "full" (~4-6 h, for the paper)
SEED               = 42              # master seed (data split, CV, models, samplers)
USE_GOOGLE_DRIVE   = True            # Colab only: store results on Drive so a disconnect cannot erase them
DRIVE_RESULTS_DIR  = "DM_BanFakeNews_results"
BERT_MODEL_NAME    = "csebuetnlp/banglabert"      # course-teacher-specified model (ELECTRA discriminator, 110M params)
EXTRA_TRANSFORMERS = []              # optional extra checkpoints to compare, e.g. ["sagorsarker/bangla-bert-base"]
# ----------------------------------------------------------------------------------------------

PROFILES = {
    # Every profile runs the *same code paths*; profiles only change the amount of computation.
    "smoke": dict(subsample=1500, inner_cv=2, grid="none", rep_folds=3, rep_repeats=1, n_boot=200,
                  bert_seeds=[42], bert_epochs=1, bert_patience=1, bert_conditions=["none", "class_weight"],
                  bert_max_train=320, bert_max_test=160, bert_bs=16, run_bert_cv=False, bert_cv_repeats=1,
                  bert_input_ablation=False, ci_auc_topk=3, group_folds=3),
    "standard": dict(subsample=None, inner_cv=3, grid="small", rep_folds=5, rep_repeats=2, n_boot=1000,
                     bert_seeds=[42, 43], bert_epochs=4, bert_patience=2,
                     bert_conditions=["none", "class_weight", "ros", "rus", "text_aug"],
                     bert_max_train=None, bert_max_test=None, bert_bs=32, run_bert_cv=False, bert_cv_repeats=1,
                     bert_input_ablation=False, ci_auc_topk=8, group_folds=5),
    "full": dict(subsample=None, inner_cv=5, grid="full", rep_folds=5, rep_repeats=3, n_boot=2000,
                 bert_seeds=[42, 43, 44], bert_epochs=5, bert_patience=2,
                 bert_conditions=["none", "class_weight", "ros", "rus", "text_aug"],
                 bert_max_train=None, bert_max_test=None, bert_bs=32, run_bert_cv=True, bert_cv_repeats=1,
                 bert_input_ablation=True, ci_auc_topk=10, group_folds=5),
}
assert PROFILE in PROFILES, f"PROFILE must be one of {list(PROFILES)}"
P = dict(PROFILES[PROFILE])

# ---- study design constants (these define the *scientific protocol*; change only on purpose) ----
CONFIG = dict(
    profile=PROFILE, seed=SEED, test_size=0.20,
    dedup=True,                                   # primary analysis on de-duplicated headlines (see §3)
    tfidf=dict(max_features=5000, ngram_range=[1, 2], sublinear_tf=True,
               token_pattern=r"[\u0980-\u09FF]+"),   # Bangla-aware tokenization (default sklearn pattern breaks Bangla words)
    tuning_metric="average_precision",            # PR-AUC of the *fake* class (threshold-free, imbalance-robust)
    threshold_objective="f1",                     # operating-point objective for threshold tuning on training-fold scores
    precision_floor=0.50,                         # policy "maximise fake recall subject to fake precision >= 0.50"
    cost_ratios=[1, 2, 5, 10, 20],                # c_FN : c_FP used for cost-sensitive model selection
    bert=dict(model=BERT_MODEL_NAME, lr=2e-5, weight_decay=0.01, warmup_ratio=0.1, val_frac=0.10,
              max_len=None, grad_clip=1.0),
    smote_k=5,
)
CONFIG.update({f"P_{k}": v for k, v in P.items()})

# ---- optional manual overrides (leave as is unless you know why) ----
RUN = dict(
    exp0_replication = True,    # §7  ML-paper protocol vs corrected protocol (explains why numbers differ from the ML paper)
    exp1_classical   = True,    # §8  tuned hold-out evaluation of all classical models x all mitigation conditions
    exp2_bert        = True,    # §9  BanglaBERT fine-tuning with the same mitigation conditions
    exp3_repeated_cv = True,    # §11 stability: repeated stratified k-fold (default hyper-parameters) + significance tests
    exp4_domain_cv   = True,    # §12 stress test: unseen news sources (domain-grouped CV)
    exp5_features    = True,    # §13 feature-representation sensitivity (word vs char n-grams)
    error_analysis   = True,    # §14
)
CONDITIONS = ["none", "class_weight", "ros", "rus", "smote", "text_aug"]
MODELS = ["Logistic Regression", "Naive Bayes", "Decision Tree", "Random Forest", "Extra Trees", "KNN", "SVM",   # ML-paper models
          "Complement NB", "LightGBM", "Stacking"]                                                                   # additions
if PROFILE == "smoke":
    MODELS = ["Logistic Regression", "Naive Bayes", "Decision Tree", "SVM", "Complement NB", "Stacking"]
N_JOBS = max(1, min(os.cpu_count() or 1, 8))          # Colab: 2 CPUs; capped so tiny jobs do not spawn many processes
print(f"PROFILE={PROFILE} | cpus={N_JOBS}\n", json.dumps(P, indent=1))

## 2. Output folders, hardware/software audit trail, reproducibility seeds

In [ ]:
def _mount_drive():
    if IN_COLAB and USE_GOOGLE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        return Path("/content/drive/MyDrive") / DRIVE_RESULTS_DIR
    return None

_drive_dir = _mount_drive()
if os.environ.get("BFN_OUT_DIR"):
    OUT_ROOT = Path(os.environ["BFN_OUT_DIR"])
elif _drive_dir is not None:
    OUT_ROOT = _drive_dir
else:
    OUT_ROOT = Path("/content/dm_results") if IN_COLAB else Path("dm_results")
OUT = OUT_ROOT / PROFILE
for sub in ["manifest", "tables", "figures", "cache", "logs"]:
    (OUT / sub).mkdir(parents=True, exist_ok=True)
_HASH_EXCLUDE = {"P_run_bert_cv", "P_bert_cv_repeats", "P_bert_input_ablation", "P_n_boot", "P_ci_auc_topk"}   # toggles that must not invalidate the cache
CONFIG_HASH = hashlib.sha1(json.dumps({k: v for k, v in CONFIG.items() if k not in _HASH_EXCLUDE},
                                      sort_keys=True, default=str).encode()).hexdigest()[:10]
CACHE = OUT / "cache" / CONFIG_HASH
CACHE.mkdir(parents=True, exist_ok=True)
print("Output folder:", OUT.resolve()); print("Cache folder :", CACHE.name, "(re-running resumes from here)")

def set_all_seeds(seed):
    random.seed(seed); np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    except Exception:
        pass
set_all_seeds(SEED)

In [ ]:
def environment_report():
    info = dict(timestamp=dt.datetime.now().isoformat(timespec="seconds"), python=platform.python_version(),
                platform=platform.platform(), cpu_count=os.cpu_count(), in_colab=IN_COLAB)
    for mod in ["numpy", "pandas", "scipy", "sklearn", "imblearn", "lightgbm", "matplotlib", "torch", "transformers", "normalizer"]:
        try:
            m = importlib.import_module(mod); info[f"pkg_{mod}"] = getattr(m, "__version__", "installed")
        except Exception:
            info[f"pkg_{mod}"] = "NOT INSTALLED"
    try:
        import torch
        info["cuda_available"] = bool(torch.cuda.is_available())
        if torch.cuda.is_available():
            info["gpu"] = torch.cuda.get_device_name(0)
            info["gpu_mem_gb"] = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
    except Exception:
        info["cuda_available"] = False
    try:
        import psutil; info["ram_gb"] = round(psutil.virtual_memory().total / 1e9, 1)
    except Exception:
        pass
    return info

ENV = environment_report()
print(json.dumps(ENV, indent=1))
if ENV.get("cuda_available") is False:
    print("\n[!] No GPU detected. Classical experiments are fine, but BanglaBERT fine-tuning will be very slow.\n"
          "    Colab: Runtime > Change runtime type > T4 GPU, then re-run from the top.")

## 3. Data loading, integrity audit and de-duplication
Dataset: **BanFakeNews** (Hossain et al., LREC 2020; Kaggle `cryptexcode/banfakenews`). We use the two *labeled* files, exactly as in the
ML paper: `LabeledAuthentic-7K.csv` (label 1 = authentic) and `LabeledFake-1K.csv` (label 0 = fake, with `F-type`).
**Internally this notebook re‑codes the labels so that `is_fake = 1` is the positive (minority) class**, which keeps every metric
call unambiguous (the ML paper used `labels=[0]` for the fake class).

In [ ]:
def find_dataset_files():
    names = ("LabeledAuthentic-7K.csv", "LabeledFake-1K.csv")
    cands = []
    if os.environ.get("BFN_DATA_DIR"): cands.append(Path(os.environ["BFN_DATA_DIR"]))
    cands += [Path("/content"), Path("/content/BanFakeNews Dataset"), Path("/content/drive/MyDrive/BanFakeNews Dataset"),
              Path("/content/drive/MyDrive/BanFakeNews"), Path("/content/drive/MyDrive/Data Mining/BanFakeNews Dataset"),
              Path("/content/drive/MyDrive/DM_BanFakeNews_data"), Path.cwd(), Path.cwd() / "BanFakeNews Dataset"]
    for d in cands:
        if all((d / n).exists() for n in names):
            return d / names[0], d / names[1]
    if IN_COLAB:                                                   # last resort: ask the user to upload the two files
        from google.colab import files
        print("Dataset not found. Please upload LabeledAuthentic-7K.csv and LabeledFake-1K.csv")
        files.upload()
        if all((Path("/content") / n).exists() for n in names):
            return Path("/content") / names[0], Path("/content") / names[1]
    raise FileNotFoundError("Could not find the two labeled CSV files. Put them in /content or in your Drive folder "
                            "'BanFakeNews Dataset' (see HOW_TO_RUN.md) or set the BFN_DATA_DIR environment variable.")

def sha256(path, n=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while chunk := f.read(n): h.update(chunk)
    return h.hexdigest()

AUTH_PATH, FAKE_PATH = find_dataset_files()
print("Using:", AUTH_PATH, "\n      ", FAKE_PATH)
raw_a = pd.read_csv(AUTH_PATH); raw_f = pd.read_csv(FAKE_PATH)

# ---- integrity checks (fail early if the wrong files were uploaded) ----
assert {"headline", "label", "domain", "category", "date", "relation", "articleID"} <= set(raw_a.columns), "unexpected columns (authentic)"
assert {"headline", "label", "F-type"} <= set(raw_f.columns), "unexpected columns (fake)"
assert set(raw_a["label"].unique()) == {1}, f"authentic file labels: {raw_a['label'].unique()}"
assert set(raw_f["label"].unique()) == {0}, f"fake file labels: {raw_f['label'].unique()}"
print(f"raw rows -> authentic: {len(raw_a):,} | fake: {len(raw_f):,}  (expected 7,202 / 1,299)")

In [ ]:
BANGLA_ONLY = re.compile(r"[^\u0980-\u09FF\s]")
def clean_bangla(t):
    """Same cleaning as the ML paper: keep Bangla-block characters (U+0980-U+09FF) + whitespace, collapse spaces."""
    return re.sub(r"\s+", " ", BANGLA_ONLY.sub("", str(t))).strip()

def build_frame():
    a = raw_a.assign(uid=lambda d: "A" + d["articleID"].astype(str), is_fake=0, ftype="Authentic")
    f = raw_f.assign(uid=lambda d: "F" + d["articleID"].astype(str), is_fake=1, ftype=raw_f["F-type"].astype(str).str.strip())
    keep = ["uid", "articleID", "domain", "date", "category", "relation", "headline", "content", "is_fake", "ftype"]
    d = pd.concat([a[keep], f[keep]], ignore_index=True)
    d["relation"] = d["relation"].astype(str).str.strip().str.lower()
    d["clean_text"] = d["headline"].map(clean_bangla)
    d["content_words"] = d["content"].astype(str).str.split().str.len()
    return d.drop(columns=["content"])

df_all = build_frame()
AUDIT = OrderedDict()
AUDIT["raw_authentic"] = int(len(raw_a)); AUDIT["raw_fake"] = int(len(raw_f))
AUDIT["missing_headline"] = int(df_all["headline"].isna().sum())
short = df_all["clean_text"].str.len() <= 3
AUDIT["dropped_too_short_after_cleaning(<=3 chars)"] = int(short.sum())
AUDIT["dropped_short_by_class"] = {"authentic": int((short & (df_all.is_fake == 0)).sum()), "fake": int((short & (df_all.is_fake == 1)).sum())}
df_clean = df_all[~short].reset_index(drop=True)                      # == ML-paper dataset (8,499 rows)
AUDIT["rows_after_cleaning(ML_paper_dataset)"] = int(len(df_clean))
AUDIT["ML_paper_class_counts"] = {"authentic": int((df_clean.is_fake == 0).sum()), "fake": int((df_clean.is_fake == 1).sum())}

# ---- conflicting labels and exact duplicates (on the cleaned headline text) ----
g = df_clean.groupby("clean_text")["is_fake"].nunique()
conflict_texts = set(g[g > 1].index)
AUDIT["headlines_with_conflicting_labels"] = int(len(conflict_texts))
dup_mask = df_clean.duplicated(subset="clean_text", keep="first")
AUDIT["duplicate_rows_(same cleaned headline, any label)"] = int(dup_mask.sum())
AUDIT["duplicate_rows_by_class"] = {"authentic": int((dup_mask & (df_clean.is_fake == 0)).sum()), "fake": int((dup_mask & (df_clean.is_fake == 1)).sum())}

if CONFIG["dedup"]:
    df = df_clean[~df_clean["clean_text"].isin(conflict_texts)]       # drop ambiguous headlines entirely
    df = df.drop_duplicates(subset="clean_text", keep="first").reset_index(drop=True)
else:
    df = df_clean.copy()
AUDIT["rows_analysed"] = int(len(df))
AUDIT["analysed_class_counts"] = {"authentic": int((df.is_fake == 0).sum()), "fake": int((df.is_fake == 1).sum())}
AUDIT["imbalance_ratio_authentic_to_fake"] = round(float((df.is_fake == 0).sum() / (df.is_fake == 1).sum()), 3)
AUDIT["majority_class_accuracy_baseline"] = round(float((df.is_fake == 0).mean()), 4)
AUDIT["fake_prevalence"] = round(float(df.is_fake.mean()), 4)
AUDIT["dataset_sha256"] = {AUTH_PATH.name: sha256(AUTH_PATH), FAKE_PATH.name: sha256(FAKE_PATH)}

# optional stratified sub-sample (smoke profile only): keeps the 5.5:1 ratio
if P["subsample"]:
    df, _ = train_test_split(df, train_size=P["subsample"], stratify=df["is_fake"], random_state=SEED)
    df = df.reset_index(drop=True)
    AUDIT["SMOKE_SUBSAMPLE_rows"] = int(len(df))
    print(f"[smoke] using a stratified sub-sample of {len(df)} rows - results are NOT for the paper")

df["domain"] = df["domain"].astype(str)
print(json.dumps(AUDIT, indent=1, default=str))
(OUT / "manifest" / "data_audit.json").write_text(json.dumps(AUDIT, indent=1, default=str), encoding="utf-8")
(OUT / "manifest" / "config.json").write_text(json.dumps(CONFIG, indent=1, default=str), encoding="utf-8")
(OUT / "manifest" / "run_manifest.json").write_text(json.dumps(ENV, indent=1), encoding="utf-8")

## 4. Exploratory data analysis (documents *what the classifiers can exploit*)
The guide asks that source bias, time sensitivity and label noise be discussed rather than assumed away. This section measures them.

In [ ]:
PALETTE = ["#0072B2", "#E69F00", "#009E73", "#D55E00", "#CC79A7", "#56B4E9", "#7F7F7F", "#000000", "#F0E442"]
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "font.size": 10, "axes.titlesize": 11, "axes.labelsize": 10,
                     "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.25,
                     "legend.frameon": False, "figure.autolayout": False})
CLASS_COLORS = {0: PALETTE[0], 1: PALETTE[3]}   # authentic = blue, fake = vermillion (colour-blind safe)

def savefig(fig, name, show=True):
    fig.savefig(OUT / "figures" / f"{name}.png", bbox_inches="tight")
    fig.savefig(OUT / "figures" / f"{name}.pdf", bbox_inches="tight")
    if show: plt.show()
    plt.close(fig)

EDA = OrderedDict()
df["hl_words"] = df["clean_text"].str.split().str.len()
fig, axs = plt.subplots(1, 3, figsize=(13, 3.6))
# (a) class distribution
cnt = df["is_fake"].value_counts().sort_index()
axs[0].bar(["Authentic", "Fake"], [cnt[0], cnt[1]], color=[CLASS_COLORS[0], CLASS_COLORS[1]], edgecolor="black", linewidth=.6)
for i, v in enumerate([cnt[0], cnt[1]]): axs[0].text(i, v + cnt.max() * .01, f"{v:,}\n({v / len(df) * 100:.1f}%)", ha="center", fontsize=9)
axs[0].set_ylim(0, cnt.max() * 1.18); axs[0].set_title("(a) Class distribution (analysed data)"); axs[0].set_ylabel("Headlines")
# (b) headline length
bins = np.arange(0, df["hl_words"].max() + 2)
for c, nm in [(0, "Authentic"), (1, "Fake")]:
    axs[1].hist(df.loc[df.is_fake == c, "hl_words"], bins=bins, alpha=.6, density=True, color=CLASS_COLORS[c], label=nm)
axs[1].set_title("(b) Headline length (words)"); axs[1].set_xlabel("Words per headline"); axs[1].set_ylabel("Density"); axs[1].legend()
u = stats.mannwhitneyu(df.loc[df.is_fake == 1, "hl_words"], df.loc[df.is_fake == 0, "hl_words"], alternative="two-sided")
EDA["headline_words_mean"] = {"authentic": round(df.loc[df.is_fake == 0, "hl_words"].mean(), 2), "fake": round(df.loc[df.is_fake == 1, "hl_words"].mean(), 2)}
EDA["headline_length_mannwhitney_p"] = float(u.pvalue)
EDA["headline_length_cliffs_delta"] = round(float(2 * u.statistic / ((df.is_fake == 1).sum() * (df.is_fake == 0).sum()) - 1), 3)
# (c) fake types
ft = df.loc[df.is_fake == 1, "ftype"].value_counts()
axs[2].bar(ft.index, ft.values, color=PALETTE[3], edgecolor="black", linewidth=.6)
for i, v in enumerate(ft.values): axs[2].text(i, v + ft.max() * .01, f"{v:,} ({v / ft.sum() * 100:.1f}%)", ha="center", fontsize=9)
axs[2].set_ylim(0, ft.max() * 1.18); axs[2].set_title("(c) Composition of the fake class (F-type)")
EDA["fake_ftype_counts"] = ft.to_dict()
fig.tight_layout(); savefig(fig, "fig01_class_length_ftype")

In [ ]:
# Source (domain) and time confounds
dom = df.groupby(["is_fake", "domain"]).size().rename("n").reset_index()
EDA["n_domains"] = {"authentic": int(dom[dom.is_fake == 0].domain.nunique()), "fake": int(dom[dom.is_fake == 1].domain.nunique())}
top4_fake_share = dom[dom.is_fake == 1].nlargest(4, "n")["n"].sum() / max(1, (df.is_fake == 1).sum())
EDA["fake_share_of_top4_domains"] = round(float(top4_fake_share), 3)
dts = pd.to_datetime(df["date"], errors="coerce", utc=True)
df["_date"] = dts
EDA["date_range_authentic"] = [str(dts[df.is_fake == 0].min()), str(dts[df.is_fake == 0].max())]
EDA["date_range_fake"] = [str(dts[df.is_fake == 1].min()), str(dts[df.is_fake == 1].max())]
EDA["unparseable_dates_fake"] = int(dts[df.is_fake == 1].isna().sum())
rel = pd.crosstab(df["is_fake"].map({0: "Authentic", 1: "Fake"}), df["relation"], normalize="index").round(4) * 100
EDA["relation_pct_by_class"] = rel.to_dict()

fig, axs = plt.subplots(1, 3, figsize=(13, 3.8))
for c, nm in [(0, "Authentic"), (1, "Fake")]:
    s = dom[dom.is_fake == c].sort_values("n", ascending=False)
    axs[0].plot(np.arange(1, len(s) + 1), s["n"].cumsum() / s["n"].sum(), marker="o", ms=3, color=CLASS_COLORS[c], label=f"{nm} ({len(s)} domains)")
axs[0].set_xlabel("Domains ranked by size"); axs[0].set_ylabel("Cumulative share of headlines"); axs[0].set_title("(a) Source concentration"); axs[0].legend()
for c, nm in [(0, "Authentic"), (1, "Fake")]:
    yrs = dts[(df.is_fake == c) & dts.notna()]
    axs[1].hist(yrs.dt.year + (yrs.dt.dayofyear - 1) / 365.25, bins=np.arange(2011, 2020.5, .25), alpha=.65, color=CLASS_COLORS[c], label=nm)
axs[1].set_xlabel("Publication date"); axs[1].set_ylabel("Headlines"); axs[1].set_title("(b) Publication time"); axs[1].legend()
rel.plot.bar(ax=axs[2], color=[PALETTE[2], PALETTE[1], PALETTE[6]][:rel.shape[1]], edgecolor="black", linewidth=.5, rot=0)
axs[2].set_ylabel("% of class"); axs[2].set_title("(c) 'relation' field (headline-body match)"); axs[2].legend(title="")
fig.tight_layout(); savefig(fig, "fig02_source_time_relation_confounds")
print(json.dumps(EDA, indent=1, default=str))
(OUT / "manifest" / "eda_summary.json").write_text(json.dumps(EDA, indent=1, default=str), encoding="utf-8")

**Reading guide for the confound figure** — authentic articles come from 22 portals over ~3 weeks of 2018, while fake articles come from
84 domains (four of them supply most rows) over 2011–2019, and the `relation` metadata field almost perfectly separates the classes.
Classifiers using only the headline text are therefore *not* guaranteed to learn veracity; they may learn source style or topic/time.
This is a limitation to be reported, and §12 stress-tests it with source-held-out folds. The `relation` field is **not** used as a feature.

## 5. Hold-out split (stratified 80/20) — the test set is used **once**

In [ ]:
tr_idx, te_idx = train_test_split(np.arange(len(df)), test_size=CONFIG["test_size"], stratify=df["is_fake"].values, random_state=SEED)
tr_idx, te_idx = np.sort(tr_idx), np.sort(te_idx)
df_tr, df_te = df.iloc[tr_idx].reset_index(drop=True), df.iloc[te_idx].reset_index(drop=True)
X_tr, y_tr = df_tr["clean_text"].values, df_tr["is_fake"].values.astype(int)
X_te, y_te = df_te["clean_text"].values, df_te["is_fake"].values.astype(int)
SPLIT = {"n_train": int(len(df_tr)), "n_test": int(len(df_te)), "train_fake": int(y_tr.sum()), "test_fake": int(y_te.sum()),
         "train_authentic": int((y_tr == 0).sum()), "test_authentic": int((y_te == 0).sum()),
         "test_majority_accuracy": round(float((y_te == 0).mean()), 4)}
print(json.dumps(SPLIT, indent=1))
pd.DataFrame({"uid": df["uid"].values, "partition": np.where(np.isin(np.arange(len(df)), te_idx), "test", "train")}).to_csv(OUT / "manifest" / "split_indices.csv", index=False)
(OUT / "manifest" / "split_summary.json").write_text(json.dumps(SPLIT, indent=1), encoding="utf-8")
assert len(set(df_tr.uid) & set(df_te.uid)) == 0
if CONFIG["dedup"]:
    assert len(set(df_tr.clean_text) & set(df_te.clean_text)) == 0, "identical headlines leaked across train/test"
print("OK: no identical headline appears in both train and test.")

## 5b. Evaluation toolkit — metrics, operating points, confidence intervals, caching
**Positive class = fake (`is_fake = 1`).** Everything below is computed from a continuous *score* (probability or SVM margin) and a
*threshold*, so the same code evaluates (i) the default operating point, (ii) a tuned threshold and (iii) cost-sensitive thresholds.

| Metric | Why it is reported |
|---|---|
| Fake precision / recall / F1 / F2 | what the ML paper optimised; F2 weights recall higher than precision |
| Authentic precision / recall / F1, macro‑F1, weighted‑F1 | guide §5 (per‑class and macro/weighted) |
| Balanced accuracy, G‑mean, MCC, Cohen's κ | single‑number summaries that are not inflated by the majority class |
| PR‑AUC (average precision), ROC‑AUC | threshold‑free ranking quality; PR‑AUC is the imbalance‑appropriate one |
| Accuracy | reported only for continuity with the ML paper (majority baseline shown alongside) |

In [ ]:
def _div(a, b):
    a = np.asarray(a, dtype=float); b = np.asarray(b, dtype=float)
    out = np.zeros(np.broadcast(a, b).shape, dtype=float)
    np.divide(a, b, out=out, where=(b != 0))
    return out

def metrics_from_counts(tp, fp, fn, tn):
    """Vectorised: works for scalars or numpy arrays of counts (used by the bootstrap)."""
    tp, fp, fn, tn = [np.asarray(x, dtype=float) for x in (tp, fp, fn, tn)]
    n = tp + fp + fn + tn
    prec = _div(tp, tp + fp); rec = _div(tp, tp + fn)
    f1 = _div(2 * prec * rec, prec + rec); f2 = _div(5 * prec * rec, 4 * prec + rec)
    spec = _div(tn, tn + fp); prec_a = _div(tn, tn + fn); f1_a = _div(2 * prec_a * spec, prec_a + spec)
    acc = _div(tp + tn, n); bal = (rec + spec) / 2
    macro = (f1 + f1_a) / 2; wf1 = _div(f1 * (tp + fn) + f1_a * (tn + fp), n)
    mcc = _div(tp * tn - fp * fn, np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn)))
    pe = _div((tp + fp) * (tp + fn) + (tn + fn) * (tn + fp), n ** 2)
    kappa = _div(acc - pe, 1 - pe)
    gm = np.sqrt(rec * spec); flag = _div(tp + fp, n)
    return dict(precision_fake=prec, recall_fake=rec, f1_fake=f1, f2_fake=f2, precision_auth=prec_a, recall_auth=spec,
                f1_auth=f1_a, macro_f1=macro, weighted_f1=wf1, accuracy=acc, balanced_acc=bal, gmean=gm, mcc=mcc,
                kappa=kappa, flag_rate=flag)

def counts(y, pred):
    y = np.asarray(y).astype(bool); p = np.asarray(pred).astype(bool)
    return int((y & p).sum()), int((~y & p).sum()), int((y & ~p).sum()), int((~y & ~p).sum())

def threshold_metrics(y, score, thr):
    tp, fp, fn, tn = counts(y, np.asarray(score) >= thr)
    m = {k: float(v) for k, v in metrics_from_counts(tp, fp, fn, tn).items()}
    m.update(tp=tp, fp=fp, fn=fn, tn=tn, thr=float(thr))
    return m

def ranking_metrics(y, score):
    y = np.asarray(y)
    if y.min() == y.max(): return dict(pr_auc=np.nan, roc_auc=np.nan)
    return dict(pr_auc=float(average_precision_score(y, score)), roc_auc=float(roc_auc_score(y, score)))

def full_metrics(y, score, thr):
    return {**threshold_metrics(y, score, thr), **ranking_metrics(y, score)}

PCT_COLS = ["precision_fake", "recall_fake", "f1_fake", "f2_fake", "precision_auth", "recall_auth", "f1_auth", "macro_f1", "weighted_f1",
            "accuracy", "balanced_acc", "gmean", "mcc", "kappa", "flag_rate", "pr_auc", "roc_auc"]
def as_pct(df_, nd=2):
    d = df_.copy(); prefs = ["", "d_", "t_", "p_", "cvd_", "cvt_", "cvp_"]
    for c in PCT_COLS:
        for pre in prefs:
            if pre + c in d.columns: d[pre + c] = (pd.to_numeric(d[pre + c], errors="coerce") * 100).round(nd)
    for c in ["cv_pooled_pr_auc"]:
        if c in d.columns: d[c] = (d[c] * 100).round(nd)
    return d

def show(obj):
    try:
        from IPython.display import display; display(obj)
    except Exception:
        print(obj)


# ---- operating points -------------------------------------------------------------------------
def default_threshold(model_name):
    return 0.0 if model_name == "SVM" else 0.5          # LinearSVC outputs margins; every other model outputs probabilities

def _pr_arrays(y, score):
    p, r, t = precision_recall_curve(y, score)
    return p[:-1], r[:-1], t                              # aligned with thresholds (score >= t is flagged fake)

def tune_threshold(y, score, objective="f1"):
    """Threshold maximising F1 (or F2) of the fake class on the given (training-side) scores."""
    p, r, t = _pr_arrays(y, score)
    if len(t) == 0: return float(np.median(score))
    beta2 = 4.0 if objective == "f2" else 1.0
    f = _div((1 + beta2) * p * r, beta2 * p + r)
    return float(t[int(np.argmax(f))])

def threshold_recall_at_precision(y, score, p_min):
    """Highest-recall threshold whose precision >= p_min (policy: 'catch as many fakes as possible, but keep alerts trustworthy')."""
    p, r, t = _pr_arrays(y, score)
    ok = p >= p_min
    if not ok.any(): return np.nan
    i = np.where(ok)[0][np.argmax(r[ok])]
    return float(t[i])

def threshold_min_cost(y, score, ratio):
    """Threshold minimising  ratio*FN + FP  (ratio = cost of a missed fake / cost of a false alarm)."""
    p, r, t = _pr_arrays(y, score)
    if len(t) == 0: return float(np.median(score))
    P = float(np.sum(y)); tp = r * P; fp = _div(tp * (1 - p), p); fn = P - tp
    return float(t[int(np.argmin(ratio * fn + fp))])

def expected_cost(y, score, thr, ratio):
    tp, fp, fn, tn = counts(y, np.asarray(score) >= thr)
    return (ratio * fn + fp) / len(y)

# ---- bootstrap confidence intervals ----------------------------------------------------------
def boot_ci_threshold_metrics(y, pred, B=1000, seed=0, keys=("precision_fake", "recall_fake", "f1_fake", "macro_f1", "mcc", "balanced_acc")):
    """Stratified bootstrap (class counts fixed) 95% percentile CI for threshold-based metrics. Vectorised."""
    y = np.asarray(y).astype(bool); pred = np.asarray(pred).astype(bool)
    rng = np.random.default_rng(seed)
    pos, neg = pred[y], pred[~y]
    ip = rng.integers(0, len(pos), (B, len(pos))); ineg = rng.integers(0, len(neg), (B, len(neg)))
    tp = pos[ip].sum(1); fn = len(pos) - tp; fp = neg[ineg].sum(1); tn = len(neg) - fp
    m = metrics_from_counts(tp, fp, fn, tn)
    return {k: (float(np.percentile(m[k], 2.5)), float(np.percentile(m[k], 97.5))) for k in keys}

def boot_ci_auc(y, score, B=500, seed=0):
    y = np.asarray(y).astype(bool); score = np.asarray(score)
    rng = np.random.default_rng(seed); sp, sn = score[y], score[~y]; out = {"pr_auc": [], "roc_auc": []}
    for _ in range(B):
        s = np.concatenate([sp[rng.integers(0, len(sp), len(sp))], sn[rng.integers(0, len(sn), len(sn))]])
        yy = np.r_[np.ones(len(sp)), np.zeros(len(sn))]
        out["pr_auc"].append(average_precision_score(yy, s)); out["roc_auc"].append(roc_auc_score(yy, s))
    return {k: (float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))) for k, v in out.items()}

def paired_bootstrap_diff(y, predA, predB, metric="f1_fake", B=2000, seed=0):
    """CI for metric(A) - metric(B) on the *same* resampled test cases (stratified). Returns (mean_diff, lo, hi, p_two_sided)."""
    y = np.asarray(y).astype(bool); A = np.asarray(predA).astype(bool); Bp = np.asarray(predB).astype(bool)
    rng = np.random.default_rng(seed); ypos, yneg = np.where(y)[0], np.where(~y)[0]
    ip = ypos[rng.integers(0, len(ypos), (B, len(ypos)))]; ineg = yneg[rng.integers(0, len(yneg), (B, len(yneg)))]
    def met(pred):
        tp = pred[ip].sum(1); fp = pred[ineg].sum(1)
        return metrics_from_counts(tp, fp, len(ypos) - tp, len(yneg) - fp)[metric]
    d = met(A) - met(Bp)
    p = 2 * min((d <= 0).mean(), (d >= 0).mean())
    return float(d.mean()), float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5)), float(min(1.0, p))

def mcnemar_exact(y, predA, predB):
    """Exact McNemar test on correctness of two classifiers (same test set). Returns (b, c, p)."""
    y = np.asarray(y).astype(bool); a = (np.asarray(predA).astype(bool) == y); b_ = (np.asarray(predB).astype(bool) == y)
    b = int((a & ~b_).sum()); c = int((~a & b_).sum())
    p = 1.0 if b + c == 0 else float(stats.binomtest(min(b, c), b + c, 0.5).pvalue)
    return b, c, p

def holm(pvals):
    p = np.asarray(pvals, dtype=float); order = np.argsort(p); m = len(p); adj = np.empty(m)
    run = 0.0
    for rank, i in enumerate(order):
        run = max(run, (m - rank) * p[i]); adj[i] = min(1.0, run)
    return adj

def nb_corrected_ttest(d, n_train, n_test):
    """Nadeau-Bengio corrected resampled t-test for paired differences d over repeated k-fold folds."""
    d = np.asarray(d, dtype=float); J = len(d)
    if J < 3 or np.allclose(d.var(ddof=1), 0): return float(d.mean()), np.nan, 1.0 if np.allclose(d, 0) else np.nan
    t = d.mean() / np.sqrt((1.0 / J + n_test / n_train) * d.var(ddof=1))
    return float(d.mean()), float(t), float(2 * stats.t.sf(abs(t), J - 1))

# ---- result cache (makes the notebook resumable after a Colab disconnect) ----------------------
def _safe(key): return re.sub(r"[^A-Za-z0-9_.+=-]+", "_", key)
def cache_has(key): return (CACHE / f"{_safe(key)}.pkl").exists()
def cache_put(key, obj):
    tmp = CACHE / f"{_safe(key)}.tmp"
    with open(tmp, "wb") as f: pickle.dump(obj, f, protocol=4)
    tmp.replace(CACHE / f"{_safe(key)}.pkl")
def cache_get(key):
    with open(CACHE / f"{_safe(key)}.pkl", "rb") as f: return pickle.load(f)

class Logger:
    def __init__(self, path): self.path = path
    def __call__(self, *a):
        s = " ".join(str(x) for x in a); print(s)
        with open(self.path, "a", encoding="utf-8") as f: f.write(f"[{dt.datetime.now():%H:%M:%S}] {s}\n")
log = Logger(OUT / "logs" / "run_log.txt")
log(f"=== run started | profile={PROFILE} | config hash={CONFIG_HASH} ===")

def fmt_time(s):
    s = int(s); return f"{s // 3600}h{(s % 3600) // 60:02d}m{s % 60:02d}s" if s >= 3600 else f"{s // 60}m{s % 60:02d}s"

# quick self-test of the metric code against scikit-learn (guards against silent bugs)
from sklearn.metrics import f1_score, matthews_corrcoef, recall_score, precision_score, balanced_accuracy_score, cohen_kappa_score
_rng = np.random.default_rng(1); _y = (_rng.random(500) < .2).astype(int); _s = _rng.random(500) * .6 + _y * .3
_m = threshold_metrics(_y, _s, .5); _p = (_s >= .5).astype(int)
assert abs(_m["f1_fake"] - f1_score(_y, _p)) < 1e-9 and abs(_m["mcc"] - matthews_corrcoef(_y, _p)) < 1e-9
assert abs(_m["recall_fake"] - recall_score(_y, _p)) < 1e-9 and abs(_m["precision_fake"] - precision_score(_y, _p)) < 1e-9
assert abs(_m["balanced_acc"] - balanced_accuracy_score(_y, _p)) < 1e-9 and abs(_m["kappa"] - cohen_kappa_score(_y, _p)) < 1e-9
assert abs(_m["macro_f1"] - f1_score(_y, _p, average="macro")) < 1e-9 and abs(_m["weighted_f1"] - f1_score(_y, _p, average="weighted")) < 1e-9
print("metric self-test passed")

## 6. Model zoo, imbalance conditions and leakage-safe pipelines

**Classifiers** — the seven from the ML paper are kept unchanged (same defaults as the baseline), plus three justified additions:

| Model | Role in the comparison | Added? |
|---|---|---|
| Logistic Regression, Linear SVM | linear models, strong on sparse high-dimensional text | ML paper |
| Multinomial NB | probabilistic baseline for count/TF-IDF features | ML paper |
| Decision Tree, Random Forest, Extra Trees | non-linear tree family / bagging / randomised ensembles | ML paper |
| KNN | instance-based baseline (suffers in high-dimensional sparse space) | ML paper |
| **Complement NB** | NB variant designed for imbalanced text; tests the ML paper's "class-prior" explanation | **new** |
| **LightGBM** | gradient boosting (guide §4.3); native sparse support; `class_weight` supported | **new** |
| **Stacking (LR + SVM + CNB → LR)** | the strongest approach reported on this dataset by the closest prior work (Hossen et al.) | **new** |
| **BanglaBERT** (§9) | contextual Bangla language model requested by the course teacher | **new** |

**Imbalance conditions** (guide §4.4). Resampling is performed **only inside the training folds** by an `imblearn.Pipeline`:

| Key | Condition |
|---|---|
| `none` | original class distribution |
| `class_weight` | `class_weight='balanced'` where the estimator supports it. For Multinomial NB the equivalent is a *uniform class prior* (`fit_prior=False`); KNN / Complement NB have no equivalent → *not applicable* (the ML paper silently re-used the baseline) |
| `ros` | random oversampling of fake training headlines to 1 : 1 |
| `rus` | random undersampling of authentic training headlines to 1 : 1 (information loss acknowledged) |
| `smote` | SMOTE (k = 5) on TF‑IDF vectors (synthetic interpolation is not a natural text operation → evaluated, not assumed) |
| `text_aug` | **text-appropriate alternative:** minority headlines are augmented at the *word* level (random deletion / adjacent swap) before vectorisation, until 1 : 1 |
| *threshold tuning* | applied **on top of every condition** (§8): the decision threshold is chosen on training-fold scores only |

In [ ]:
class TextAugmentOverSampler(BaseEstimator):
    """imblearn-style sampler operating on raw text: grows the minority class to 1:1 with label-preserving word-level noise
    (random word deletion or swap of two adjacent words). Applied only to training data inside the Pipeline."""
    def __init__(self, random_state=42, p_delete=0.15):
        self.random_state = random_state; self.p_delete = p_delete
    def fit(self, X, y=None): return self
    def _augment(self, text, rng):
        w = text.split()
        if len(w) >= 4 and rng.random() < 0.5:                                  # random deletion (keep >= 3 words)
            keep = [t for t in w if rng.random() > self.p_delete]
            w = keep if len(keep) >= 3 else w
        elif len(w) >= 2:                                                       # swap two adjacent words
            i = int(rng.integers(0, len(w) - 1)); w[i], w[i + 1] = w[i + 1], w[i]
        return " ".join(w)
    def fit_resample(self, X, y):
        X = np.asarray(X, dtype=object); y = np.asarray(y); rng = np.random.default_rng(self.random_state)
        cls, cnt = np.unique(y, return_counts=True); target = cnt.max(); Xs, ys = [X], [y]
        for c, n in zip(cls, cnt):
            if n >= target: continue
            pool = X[y == c]; pick = rng.integers(0, len(pool), target - n)
            Xs.append(np.array([self._augment(pool[i], rng) for i in pick], dtype=object)); ys.append(np.full(target - n, c))
        return np.concatenate(Xs), np.concatenate(ys)

def make_vectorizer(kind="word_paper"):
    t = CONFIG["tfidf"]
    if kind == "word_paper":          # the ML paper's setting + Bangla-aware tokenizer
        return TfidfVectorizer(max_features=t["max_features"], ngram_range=tuple(t["ngram_range"]), sublinear_tf=t["sublinear_tf"], token_pattern=t["token_pattern"])
    if kind == "word_default_tokenizer":   # reproduces the ML paper's (flawed) tokenisation
        return TfidfVectorizer(max_features=t["max_features"], ngram_range=tuple(t["ngram_range"]), sublinear_tf=t["sublinear_tf"])
    if kind == "word_full":           # no vocabulary cap (min_df=2)
        return TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, min_df=2, token_pattern=t["token_pattern"])
    if kind == "char_wb":             # character n-grams (robust to Bangla morphology / spelling variation)
        return TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), sublinear_tf=True, min_df=2, max_features=100000)
    raise ValueError(kind)

def make_classifier(name, class_weight=False, seed=SEED, paper_mode=False):
    cw = "balanced" if class_weight else None
    if name == "Logistic Regression": return LogisticRegression(max_iter=1000, random_state=seed, class_weight=cw)
    if name == "Naive Bayes":         return MultinomialNB(fit_prior=not (class_weight and not paper_mode))   # uniform prior = NB's class-weight analogue
    if name == "Complement NB":       return ComplementNB()
    if name == "Decision Tree":       return DecisionTreeClassifier(random_state=seed, class_weight=cw)
    if name == "Random Forest":       return RandomForestClassifier(n_estimators=100, random_state=seed, class_weight=cw, n_jobs=1)
    if name == "Extra Trees":         return ExtraTreesClassifier(n_estimators=100, random_state=seed, class_weight=cw, n_jobs=1)
    if name == "KNN":                 return KNeighborsClassifier(n_neighbors=5)
    if name == "SVM":                 return LinearSVC(max_iter=2000, random_state=seed, class_weight=cw)
    if name == "LightGBM":            return lgb.LGBMClassifier(n_estimators=300, learning_rate=0.1, num_leaves=31, class_weight=cw, random_state=seed, n_jobs=1, verbosity=-1)
    if name == "Stacking":
        return StackingClassifier(
            estimators=[("lr", LogisticRegression(max_iter=1000, random_state=seed, class_weight=cw)),
                        ("svm", LinearSVC(max_iter=2000, random_state=seed, class_weight=cw)),
                        ("cnb", ComplementNB())],
            final_estimator=LogisticRegression(max_iter=1000, random_state=seed, class_weight=cw),
            cv=3, stack_method="auto", n_jobs=1)
    raise ValueError(name)

SUPPORTS_CW = {m: True for m in MODELS}; SUPPORTS_CW.update({"KNN": False, "Complement NB": False})      # NB uses the uniform-prior analogue
def applicable(model, cond):
    return not (cond == "class_weight" and not SUPPORTS_CW[model])

def build_pipeline(model, cond, feat="word_paper", seed=SEED, paper_mode=False):
    """TF-IDF is *inside* the pipeline => it is fitted on the training fold only. Samplers see training data only."""
    steps = []
    if cond == "text_aug": steps.append(("aug", TextAugmentOverSampler(random_state=seed)))
    steps.append(("tfidf", make_vectorizer(feat)))
    if cond == "ros":   steps.append(("sampler", RandomOverSampler(random_state=seed)))
    if cond == "rus":   steps.append(("sampler", RandomUnderSampler(random_state=seed)))
    if cond == "smote": steps.append(("sampler", SMOTE(k_neighbors=CONFIG["smote_k"], random_state=seed)))
    steps.append(("clf", make_classifier(model, class_weight=(cond == "class_weight"), seed=seed, paper_mode=paper_mode)))
    return ImbPipeline(steps)

def get_score(pipe, X, model):
    """Continuous fake-class score: probability for probabilistic models, margin for the linear SVM."""
    if model == "SVM": return pipe.decision_function(X)
    return pipe.predict_proba(X)[:, 1]

def score_method(model): return "decision_function" if model == "SVM" else "predict_proba"

# ---- hyper-parameter grids (the ML-paper default is always inside the grid, so tuning can never be worse than the default in CV) ----
GRIDS_FULL = {
    "Logistic Regression": {"clf__C": [0.1, 1.0, 10.0]},
    "Naive Bayes":         {"clf__alpha": [0.1, 0.5, 1.0]},
    "Complement NB":       {"clf__alpha": [0.1, 0.5, 1.0]},
    "Decision Tree":       {"clf__max_depth": [None, 30], "clf__min_samples_leaf": [1, 3]},
    "Random Forest":       {"clf__min_samples_leaf": [1, 2], "clf__max_features": ["sqrt", "log2"]},
    "Extra Trees":         {"clf__min_samples_leaf": [1, 2], "clf__max_features": ["sqrt", "log2"]},
    "KNN":                 {"clf__n_neighbors": [3, 5, 11], "clf__weights": ["uniform", "distance"]},
    "SVM":                 {"clf__C": [0.1, 0.5, 1.0]},
    "LightGBM":            {"clf__learning_rate": [0.05, 0.1], "clf__num_leaves": [15, 31]},
    "Stacking":            {},
}
GRIDS_SMALL = {
    "Logistic Regression": {"clf__C": [1.0, 10.0]},
    "Naive Bayes":         {"clf__alpha": [0.1, 1.0]},
    "Complement NB":       {"clf__alpha": [0.1, 1.0]},
    "Decision Tree":       {"clf__min_samples_leaf": [1, 3]},
    "Random Forest":       {"clf__min_samples_leaf": [1, 2]},
    "Extra Trees":         {"clf__min_samples_leaf": [1, 2]},
    "KNN":                 {"clf__n_neighbors": [5, 11]},
    "SVM":                 {"clf__C": [0.1, 1.0]},
    "LightGBM":            {"clf__learning_rate": [0.05, 0.1]},
    "Stacking":            {},
}
def grid_for(model):
    g = {"full": GRIDS_FULL, "small": GRIDS_SMALL, "none": {}}[P["grid"]].get(model, {})
    if model == "Stacking": g = {}                       # stacking is evaluated with its fixed, documented configuration
    return g

# ---- smoke test of every (model, condition) pipeline on a tiny sample so errors surface immediately ----
def _pipeline_selftest():
    rs = np.random.RandomState(0); idx = rs.choice(len(X_tr), size=min(400, len(X_tr)), replace=False)
    Xs, ys = X_tr[idx], y_tr[idx]
    bad = []
    for m in MODELS:
        for c in CONDITIONS:
            if not applicable(m, c): continue
            try:
                pipe = build_pipeline(m, c).fit(Xs, ys); s = get_score(pipe, Xs[:50], m); assert np.isfinite(s).all()
            except Exception as e:
                bad.append((m, c, repr(e)[:160]))
    if bad:
        for b in bad: print("PIPELINE FAILURE:", b)
        raise RuntimeError("Some pipelines failed their self-test (see above).")
    print(f"pipeline self-test passed for {sum(applicable(m, c) for m in MODELS for c in CONDITIONS)} (model, condition) combinations")
_pipeline_selftest()

## 7. Experiment 0 — Why do these numbers differ from the ML paper? (replication and protocol corrections)
The ML paper's pipeline is replicated first (**variant A**), then corrections are added one at a time so each effect is measured:

| Variant | Change relative to the previous variant |
|---|---|
| **A. ML-paper protocol** | 8,499 headlines incl. duplicates · TF‑IDF fitted on the *whole corpus* before the split · scikit‑learn default tokenizer · default hyper‑parameters |
| **B. + de-duplication** | exact duplicate / conflicting headlines removed (they can sit in both train and test) |
| **C. + train-only TF‑IDF** | vectorizer fitted on the training split only (guide §4.2) |
| **D. + Bangla-aware tokenizer** | `token_pattern=[\u0980-\u09FF]+`: scikit-learn's default `\b\w\w+\b` treats Bangla vowel signs (matras) as non-word characters and shreds words, e.g. *রাজধানীতে বিশেষ অভিযানে আটক ৪৩* → `['জধ','অভ','আটক','৪৩']` |

Classifiers: the ML paper's seven models, conditions: none / class weights / random oversampling, default hyper-parameters, single split.

In [ ]:
EXP0_MODELS = [m for m in ["Logistic Regression", "Naive Bayes", "Decision Tree", "Random Forest", "Extra Trees", "KNN", "SVM"] if m in MODELS]
EXP0_CONDS = ["none", "class_weight", "ros"]

def _fit_score_prefit(model, cond, Xtr_, ytr_, Xte_, seed=SEED):
    """Paper-style evaluation on already-vectorised matrices."""
    clf = make_classifier(model, class_weight=(cond == "class_weight"), seed=seed, paper_mode=True)
    if cond == "ros": Xtr_, ytr_ = RandomOverSampler(random_state=seed).fit_resample(Xtr_, ytr_)
    clf.fit(Xtr_, ytr_)
    s = clf.decision_function(Xte_) if model == "SVM" else clf.predict_proba(Xte_)[:, 1]
    return s

def run_exp0():
    rows = []
    t0 = time.time()
    dA = df_clean if not P["subsample"] else df_clean.sample(n=P["subsample"], random_state=SEED)
    frames = {"A_paper_protocol": dA, "B_dedup": df, "C_dedup_train_only_tfidf": df, "D_plus_bangla_tokenizer": df}
    for variant, d in frames.items():
        key = f"exp0__{variant}"
        if cache_has(key): rows += cache_get(key); continue
        res = []
        y = d["is_fake"].values.astype(int); X = d["clean_text"].values
        tok_kind = "word_paper" if variant.startswith("D") else "word_default_tokenizer"
        if variant.startswith("A") or variant.startswith("B"):       # vectorizer fitted on the full corpus BEFORE splitting (as in the ML paper)
            vec = make_vectorizer(tok_kind); Xv = vec.fit_transform(X)
            Xtr_, Xte_, ytr_, yte_ = train_test_split(Xv, y, test_size=CONFIG["test_size"], random_state=SEED, stratify=y)
        else:                                                         # proper order: split first, fit on train only
            Xt, Xe, ytr_, yte_ = train_test_split(X, y, test_size=CONFIG["test_size"], random_state=SEED, stratify=y)
            vec = make_vectorizer(tok_kind); Xtr_ = vec.fit_transform(Xt); Xte_ = vec.transform(Xe)
        for m in EXP0_MODELS:
            for c in EXP0_CONDS:
                s = _fit_score_prefit(m, c, Xtr_, ytr_, Xte_)
                met = full_metrics(yte_, s, default_threshold(m))
                res.append(dict(variant=variant, model=m, condition=c, n_train=int(Xtr_.shape[0]), n_test=int(len(yte_)), n_fake_test=int(yte_.sum()), **met))
        cache_put(key, res); rows += res
        log(f"[exp0] {variant} done ({fmt_time(time.time() - t0)})")
    return pd.DataFrame(rows)

if RUN["exp0_replication"]:
    EXP0 = run_exp0()
    EXP0.to_csv(OUT / "tables" / "exp0_replication_all_metrics.csv", index=False)
    piv = EXP0.pivot_table(index=["model", "condition"], columns="variant", values="recall_fake").mul(100).round(2)
    piv_f1 = EXP0.pivot_table(index=["model", "condition"], columns="variant", values="f1_fake").mul(100).round(2)
    piv_ap = EXP0.pivot_table(index=["model", "condition"], columns="variant", values="pr_auc").mul(100).round(2)
    piv.to_csv(OUT / "tables" / "exp0_fake_recall_by_variant.csv"); piv_f1.to_csv(OUT / "tables" / "exp0_fake_f1_by_variant.csv"); piv_ap.to_csv(OUT / "tables" / "exp0_prauc_by_variant.csv")
    print("Fake-class RECALL (%) by protocol variant"); display(piv) if "display" in globals() else print(piv)
    print("\nFake-class PR-AUC (%) by protocol variant"); display(piv_ap) if "display" in globals() else print(piv_ap)
    # figure: effect of each correction on PR-AUC (threshold-free) and fake-F1
    fig, axs = plt.subplots(1, 2, figsize=(13, 3.8), sharey=False)
    for ax, tab, ttl in [(axs[0], piv_ap, "PR-AUC of the fake class (%)"), (axs[1], piv_f1, "Fake-class F1 (%) at default threshold")]:
        t = tab.xs("none", level="condition"); vs = list(t.columns); xs = np.arange(len(t)); w = 0.8 / len(vs)
        for j, v in enumerate(vs): ax.bar(xs + j * w, t[v].values, w, label=v.replace("_", " "), color=PALETTE[j], edgecolor="black", linewidth=.4)
        ax.set_xticks(xs + 0.4 - w / 2); ax.set_xticklabels(t.index, rotation=25, ha="right"); ax.set_title(ttl + " — no mitigation")
    axs[0].legend(fontsize=7, ncol=2); fig.tight_layout(); savefig(fig, "fig03_exp0_protocol_corrections")
else:
    EXP0 = None

## 8. Experiment 1 — Tuned hold-out evaluation of all classical models × all imbalance conditions
Protocol for **every** (model, condition) pair — this is the guide's leakage-free order, executed literally:

1. Work on the **training split only**. Hyper-parameters are chosen by stratified inner CV on the training split, scoring the
   PR-AUC of the fake class (threshold-free, so it is not biased towards a particular resampling strategy).
   TF‑IDF and resampling are re-fitted inside each fold (they live in the `Pipeline`).
2. Out-of-fold (OOF) training scores of the best configuration are used to (a) report **CV-based** performance and
   (b) pick decision thresholds (F1-optimal, precision-floor policy, cost-optimal) — never the test set.
3. The finalised model is fitted on the whole training split and scored **once** on the untouched test split.

Result files: `tables/exp1_*.csv`. Resumable: finished configurations are cached.

In [ ]:
def tune_and_evaluate(model, cond, Xtr_, ytr_, Xte_, yte_, inner_cv, seed=SEED, feat="word_paper"):
    t0 = time.time()
    pipe = build_pipeline(model, cond, feat=feat, seed=seed)
    cv = StratifiedKFold(n_splits=inner_cv, shuffle=True, random_state=seed)
    grid = grid_for(model)
    if grid:
        gs = GridSearchCV(pipe, grid, scoring=CONFIG["tuning_metric"], cv=cv, n_jobs=N_JOBS, refit=True)
        gs.fit(Xtr_, ytr_)
        best, params, cv_mean_ap = gs.best_estimator_, gs.best_params_, float(gs.best_score_)
        template = clone(gs.best_estimator_)
    else:
        params, cv_mean_ap, template = {}, np.nan, clone(pipe)
        best = clone(pipe).fit(Xtr_, ytr_)
    oof = cross_val_predict(template, Xtr_, ytr_, cv=cv, method=score_method(model), n_jobs=N_JOBS)
    oof = oof[:, 1] if oof.ndim == 2 else oof
    s_te = get_score(best, Xte_, model)

    thr0 = default_threshold(model)
    thr_f1 = tune_threshold(ytr_, oof, CONFIG["threshold_objective"])
    thr_pol = threshold_recall_at_precision(ytr_, oof, CONFIG["precision_floor"])
    rec = dict(model=model, condition=cond, feat=feat, seed=seed, params=params, n_train=int(len(ytr_)), n_test=int(len(yte_)),
               cv_mean_fold_ap=cv_mean_ap, cv_pooled_ap=float(average_precision_score(ytr_, oof)), fit_seconds=time.time() - t0,
               thr_default=thr0, thr_f1=thr_f1, thr_policy=thr_pol)
    rec["test_default"] = full_metrics(yte_, s_te, thr0)
    rec["test_tuned"] = full_metrics(yte_, s_te, thr_f1)
    rec["oof_default"] = full_metrics(ytr_, oof, thr0)
    rec["oof_tuned"] = full_metrics(ytr_, oof, thr_f1)
    rec["test_policy"] = full_metrics(yte_, s_te, thr_pol) if np.isfinite(thr_pol) else None
    rec["oof_policy"] = full_metrics(ytr_, oof, thr_pol) if np.isfinite(thr_pol) else None
    rec["cost"] = {}
    for r in CONFIG["cost_ratios"]:
        th = threshold_min_cost(ytr_, oof, r)
        rec["cost"][r] = dict(thr=th, oof_cost=expected_cost(ytr_, oof, th, r), test_cost=expected_cost(yte_, s_te, th, r),
                              test_recall=threshold_metrics(yte_, s_te, th)["recall_fake"], test_precision=threshold_metrics(yte_, s_te, th)["precision_fake"],
                              oof_default_cost=expected_cost(ytr_, oof, thr0, r), test_default_cost=expected_cost(yte_, s_te, thr0, r))
    arrays = dict(oof=oof.astype(np.float32), oof_y=np.asarray(ytr_).astype(np.int8), test_score=np.asarray(s_te, dtype=np.float32), test_y=np.asarray(yte_).astype(np.int8))
    return rec, arrays

def run_exp1():
    todo = [(m, c) for m in MODELS for c in CONDITIONS if applicable(m, c)]
    t_start = time.time(); results = {}
    for i, (m, c) in enumerate(todo, 1):
        key = f"exp1__{m}__{c}"
        if cache_has(key):
            results[(m, c)] = cache_get(key); continue
        try:
            rec, arr = tune_and_evaluate(m, c, X_tr, y_tr, X_te, y_te, P["inner_cv"])
        except Exception as e:                               # a failed configuration must not kill an 1-hour run; it is logged & reported
            log(f"[exp1] FAILED {m} / {c}: {e!r}"); continue
        cache_put(key, (rec, arr)); results[(m, c)] = (rec, arr)
        td = rec["test_default"]
        log(f"[exp1] {i:>2}/{len(todo)} {m:<20} {c:<13} | {rec['fit_seconds']:5.1f}s | test@default: F1f={td['f1_fake']*100:5.1f} Rf={td['recall_fake']*100:5.1f} "
            f"Pf={td['precision_fake']*100:5.1f} PR-AUC={td['pr_auc']*100:5.1f} | elapsed {fmt_time(time.time() - t_start)}")
    return results

EXP1 = run_exp1() if RUN["exp1_classical"] else {}
print(f"\nExperiment 1 finished: {len(EXP1)} configurations")

In [ ]:
def flatten(rec, family="classical"):
    out = dict(family=family, model=rec["model"], condition=rec["condition"], seed=rec.get("seed"), params=json.dumps(rec.get("params", {}), default=str),
               cv_pooled_pr_auc=rec.get("cv_pooled_ap"), fit_seconds=rec.get("fit_seconds"), thr_default=rec["thr_default"], thr_tuned=rec["thr_f1"], thr_policy=rec["thr_policy"])
    for pref, key in [("d_", "test_default"), ("t_", "test_tuned"), ("p_", "test_policy"), ("cvd_", "oof_default"), ("cvt_", "oof_tuned"), ("cvp_", "oof_policy")]:
        if rec.get(key):
            for k, v in rec[key].items(): out[pref + k] = v
    out["pr_auc"] = rec["test_default"]["pr_auc"]; out["roc_auc"] = rec["test_default"]["roc_auc"]
    for r, cdict in rec["cost"].items():
        out[f"oofcost_r{r}"] = cdict["oof_cost"]; out[f"testcost_r{r}"] = cdict["test_cost"]; out[f"testcost_default_r{r}"] = cdict["test_default_cost"]
        out[f"cost_thr_r{r}"] = cdict["thr"]
    return out

CLASSICAL = pd.DataFrame([flatten(rec) for (rec, _) in EXP1.values()]) if EXP1 else pd.DataFrame()
if len(CLASSICAL):
    order = {m: i for i, m in enumerate(MODELS)}; corder = {c: i for i, c in enumerate(CONDITIONS)}
    CLASSICAL = CLASSICAL.sort_values(by=["model", "condition"], key=lambda s: s.map(order if s.name == "model" else corder)).reset_index(drop=True)
    as_pct(CLASSICAL).to_csv(OUT / "tables" / "exp1_classical_all_configs.csv", index=False)
    show(as_pct(CLASSICAL)[["model", "condition", "params", "cv_pooled_pr_auc", "d_f1_fake", "d_recall_fake", "d_precision_fake", "d_macro_f1", "d_mcc", "pr_auc", "roc_auc"]].head(15))

## 9. Experiment 2 — BanglaBERT (`csebuetnlp/banglabert`)
**Model.** BanglaBERT is a monolingual Bangla ELECTRA‑base discriminator (~110 M parameters, replaced‑token‑detection pre‑training on a
~27 GB Bangla corpus; Bhattacharjee et al., Findings of NAACL 2022). It expects text normalised with the authors' `normalizer` package,
which is applied here.

**Protocol (kept as close as possible to the classical models)**
* same 80/20 split and the same test headlines; the *training* split is divided 90/10 into fit/validation (stratified) — the validation
  part is used for **early stopping, model selection and threshold tuning**, never the test set;
* same imbalance conditions where meaningful: `none`, `class_weight` (weighted cross-entropy), `ros` (minority resampled each epoch to 1:1),
  `rus` (fixed random majority subset, 1:1), `text_aug` (word-level augmented minority copies, 1:1). SMOTE is **not applicable** to token
  inputs and is therefore omitted for BanglaBERT;
* several random seeds per condition → mean ± std (fine-tuning variance is large for small datasets);
* custom, version‑independent PyTorch loop: AdamW, linear warm‑up/decay, fp16 on GPU, gradient clipping, best epoch by validation PR‑AUC.

In [ ]:
BERT_AVAILABLE = True
try:
    import torch, gc
    from torch import nn
    from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
    from transformers import logging as hf_logging
    hf_logging.set_verbosity_error()
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
except Exception as e:
    BERT_AVAILABLE = False; print("PyTorch/transformers unavailable -> BanglaBERT experiments disabled:", repr(e))

try:
    from normalizer import normalize as _bn_normalize
    NORMALIZER = "csebuetnlp/normalizer"
    def bn_normalize(t): return _bn_normalize(t)
except Exception:
    import unicodedata
    NORMALIZER = "unicodedata.NFKC fallback (csebuetnlp/normalizer not installed)"
    def bn_normalize(t): return re.sub(r"\s+", " ", unicodedata.normalize("NFKC", str(t))).strip()
print("Text normalizer:", NORMALIZER)
if BERT_AVAILABLE: print("Device:", DEVICE, "|", ENV.get("gpu", "no GPU"))

def bert_texts(frame, variant="clean"):
    src = frame["clean_text"] if variant == "clean" else frame["headline"].astype(str)
    return np.array([bn_normalize(t) for t in src], dtype=object)

In [ ]:
if BERT_AVAILABLE and (RUN["exp2_bert"] or P["run_bert_cv"]):
    # stratified sub-selection used only by the smoke profile (tiny, fast)
    def _sub(idx_pool, y, n, seed):
        if n is None or n >= len(idx_pool): return idx_pool
        keep, _ = train_test_split(np.arange(len(idx_pool)), train_size=n, stratify=y[idx_pool], random_state=seed)
        return idx_pool[np.sort(keep)]
    B_TR = _sub(np.arange(len(X_tr)), y_tr, P["bert_max_train"], SEED)
    B_TE = _sub(np.arange(len(X_te)), y_te, P["bert_max_test"], SEED)
    _tok_probe = AutoTokenizer.from_pretrained(BERT_MODEL_NAME)
    _txt_probe = bert_texts(df_tr.iloc[B_TR], "clean")
    _lens = np.array([len(x) for x in _tok_probe(list(_txt_probe), truncation=True, max_length=512)["input_ids"]])
    CONFIG["bert"]["max_len"] = int(min(128, max(32, int(math.ceil((np.percentile(_lens, 99) + 2) / 8) * 8))))
    print(f"token length of training headlines: median={np.median(_lens):.0f}, p99={np.percentile(_lens, 99):.0f}, max={_lens.max()} -> max_len={CONFIG['bert']['max_len']}")
    del _tok_probe

def _predict_probs(model, enc, tok, bs=128):
    model.eval(); out = []
    with torch.no_grad():
        for i in range(0, len(enc["input_ids"]), bs):
            b = tok.pad({k: v[i:i + bs] for k, v in enc.items()}, return_tensors="pt").to(DEVICE)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=(DEVICE.type == "cuda")):
                logits = model(**b).logits
            out.append(torch.softmax(logits.float(), -1)[:, 1].cpu().numpy())
    return np.concatenate(out)

def _batches(enc, tok, idx, y, bs):
    for i in range(0, len(idx), bs):
        j = idx[i:i + bs]
        b = tok.pad({k: [v[t] for t in j] for k, v in enc.items()}, return_tensors="pt")
        yield b.to(DEVICE), torch.tensor(y[j], dtype=torch.long, device=DEVICE)

def finetune_transformer(model_id, cond, seed, tr_text, tr_y, va_text, va_y, te_text, te_y, epochs, bs, patience, max_len):
    """Fine-tune one transformer under one imbalance condition. Returns (record, arrays)."""
    t0 = time.time(); set_all_seeds(seed); rng = np.random.default_rng(seed)
    tok = AutoTokenizer.from_pretrained(model_id)
    tr_text = np.asarray(tr_text, dtype=object); tr_y = np.asarray(tr_y).astype(int)
    if cond == "text_aug":
        tr_text, tr_y = TextAugmentOverSampler(random_state=seed).fit_resample(tr_text, tr_y)
    if cond == "rus":
        mi = np.where(tr_y == 1)[0]; ma = np.where(tr_y == 0)[0]
        keep = np.sort(np.concatenate([mi, rng.choice(ma, size=len(mi), replace=False)])); tr_text, tr_y = tr_text[keep], tr_y[keep]
    enc = lambda texts: tok(list(texts), truncation=True, max_length=max_len)
    e_tr, e_va, e_te = enc(tr_text), enc(va_text), enc(te_text)
    model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2).to(DEVICE)
    nd = ["bias", "LayerNorm.weight", "LayerNorm.bias"]
    groups = [{"params": [p for n, p in model.named_parameters() if not any(x in n for x in nd)], "weight_decay": CONFIG["bert"]["weight_decay"]},
              {"params": [p for n, p in model.named_parameters() if any(x in n for x in nd)], "weight_decay": 0.0}]
    opt = torch.optim.AdamW(groups, lr=CONFIG["bert"]["lr"])
    mi, ma = np.where(tr_y == 1)[0], np.where(tr_y == 0)[0]
    def epoch_indices():
        if cond == "ros":                                              # minority re-sampled with replacement to 1:1 each epoch
            idx = np.concatenate([ma, rng.choice(mi, size=len(ma), replace=True)])
        else:
            idx = np.arange(len(tr_y))
        return rng.permutation(idx)
    steps_per_epoch = int(math.ceil(len(epoch_indices()) / bs)); total = steps_per_epoch * epochs
    sched = get_linear_schedule_with_warmup(opt, int(CONFIG["bert"]["warmup_ratio"] * total), total)
    wts = None
    if cond == "class_weight":
        n = len(tr_y); wts = torch.tensor([n / (2 * (tr_y == 0).sum()), n / (2 * (tr_y == 1).sum())], dtype=torch.float, device=DEVICE)
    lossf = nn.CrossEntropyLoss(weight=wts)
    use_amp = DEVICE.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp) if hasattr(torch.amp, "GradScaler") else torch.cuda.amp.GradScaler(enabled=use_amp)
    hist, best = [], dict(ap=-2, epoch=0, state=None); bad = 0
    va_y = np.asarray(va_y).astype(int)
    for ep in range(1, epochs + 1):
        model.train(); run, nb = 0.0, 0
        for xb, yb in _batches(e_tr, tok, epoch_indices(), tr_y, bs):
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=use_amp):
                logits = model(**xb).logits
            loss = lossf(logits.float(), yb)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), CONFIG["bert"]["grad_clip"]); scaler.step(opt); scaler.update(); sched.step()
            run += float(loss.item()); nb += 1
        pv = _predict_probs(model, e_va, tok)
        vloss = float(-np.mean(va_y * np.log(np.clip(pv, 1e-7, 1)) + (1 - va_y) * np.log(np.clip(1 - pv, 1e-7, 1))))
        vap = float(average_precision_score(va_y, pv)) if va_y.min() != va_y.max() else float("nan")
        hist.append(dict(epoch=ep, train_loss=run / max(nb, 1), val_loss=vloss, val_pr_auc=vap, val_f1_at_0p5=threshold_metrics(va_y, pv, .5)["f1_fake"]))
        if np.nan_to_num(vap, nan=-1) > best["ap"]:
            best = dict(ap=np.nan_to_num(vap, nan=-1), epoch=ep, state={k: v.detach().cpu().clone() for k, v in model.state_dict().items()}); bad = 0
        else:
            bad += 1
            if bad >= patience: break
    model.load_state_dict(best["state"]); model.to(DEVICE)
    pv = _predict_probs(model, e_va, tok); pt = _predict_probs(model, e_te, tok)
    del model, opt, sched, best["state"]; gc.collect()
    if DEVICE.type == "cuda": torch.cuda.empty_cache()
    return dict(val_probs=pv, test_probs=pt, history=hist, best_epoch=best["epoch"], epochs_run=len(hist), seconds=time.time() - t0, n_train_effective=int(len(tr_y)))

def make_transformer_record(name, cond, seed, out, va_y, te_y, extra_params=None):
    """Same record layout as the classical models. 'oof_*' fields hold *validation-split* results for transformers."""
    thr0 = 0.5; pv, pt = out["val_probs"], out["test_probs"]
    thr_f1 = tune_threshold(va_y, pv, CONFIG["threshold_objective"]); thr_pol = threshold_recall_at_precision(va_y, pv, CONFIG["precision_floor"])
    rec = dict(model=name, condition=cond, feat="transformer", seed=seed, params=dict(best_epoch=out["best_epoch"], epochs_run=out["epochs_run"], **(extra_params or {})),
               n_train=out["n_train_effective"], n_test=int(len(te_y)), cv_mean_fold_ap=np.nan, cv_pooled_ap=float(average_precision_score(va_y, pv)),
               fit_seconds=out["seconds"], thr_default=thr0, thr_f1=thr_f1, thr_policy=thr_pol, history=out["history"])
    rec["test_default"] = full_metrics(te_y, pt, thr0); rec["test_tuned"] = full_metrics(te_y, pt, thr_f1)
    rec["oof_default"] = full_metrics(va_y, pv, thr0); rec["oof_tuned"] = full_metrics(va_y, pv, thr_f1)
    rec["test_policy"] = full_metrics(te_y, pt, thr_pol) if np.isfinite(thr_pol) else None
    rec["oof_policy"] = full_metrics(va_y, pv, thr_pol) if np.isfinite(thr_pol) else None
    rec["cost"] = {}
    for r in CONFIG["cost_ratios"]:
        th = threshold_min_cost(va_y, pv, r); tm = threshold_metrics(te_y, pt, th)
        rec["cost"][r] = dict(thr=th, oof_cost=expected_cost(va_y, pv, th, r), test_cost=expected_cost(te_y, pt, th, r), test_recall=tm["recall_fake"],
                              test_precision=tm["precision_fake"], oof_default_cost=expected_cost(va_y, pv, thr0, r), test_default_cost=expected_cost(te_y, pt, thr0, r))
    arrays = dict(oof=pv.astype(np.float32), oof_y=np.asarray(va_y).astype(np.int8), test_score=pt.astype(np.float32), test_y=np.asarray(te_y).astype(np.int8))
    return rec, arrays

In [ ]:
def run_exp2(model_id=BERT_MODEL_NAME, display_name="BanglaBERT", variant="clean", conditions=None, seeds=None):
    conditions = conditions or P["bert_conditions"]; seeds = seeds or P["bert_seeds"]
    # fixed 90/10 fit/validation split of the TRAINING split (same for every condition and seed)
    fit_i, val_i = train_test_split(np.arange(len(B_TR)), test_size=CONFIG["bert"]["val_frac"], stratify=y_tr[B_TR], random_state=SEED)
    tr_rows, va_rows = B_TR[fit_i], B_TR[val_i]
    T_fit = bert_texts(df_tr.iloc[tr_rows], variant); T_val = bert_texts(df_tr.iloc[va_rows], variant); T_te = bert_texts(df_te.iloc[B_TE], variant)
    y_fit, y_val, y_tst = y_tr[tr_rows], y_tr[va_rows], y_te[B_TE]
    log(f"[exp2] {display_name} ({variant} input): fit={len(y_fit)} (fake {y_fit.sum()}), val={len(y_val)} (fake {y_val.sum()}), test={len(y_tst)} (fake {y_tst.sum()})")
    results = {}; t_start = time.time(); todo = [(c, s) for c in conditions for s in seeds]
    for i, (c, s) in enumerate(todo, 1):
        key = f"exp2__{display_name}__{variant}__{c}__{s}__e{P['bert_epochs']}__L{CONFIG['bert']['max_len']}"
        if cache_has(key): results[(c, s)] = cache_get(key); continue
        try:
            out = finetune_transformer(model_id, c, s, T_fit, y_fit, T_val, y_val, T_te, y_tst, P["bert_epochs"], P["bert_bs"], P["bert_patience"], CONFIG["bert"]["max_len"])
        except Exception as e:
            log(f"[exp2] FAILED {display_name}/{c}/seed{s}: {e!r}"); continue
        rec, arr = make_transformer_record(display_name if variant == "clean" else f"{display_name} [{variant} input]", c, s, out, y_val, y_tst,
                                           dict(model_id=model_id, input=variant, max_len=CONFIG["bert"]["max_len"], lr=CONFIG["bert"]["lr"], bs=P["bert_bs"]))
        cache_put(key, (rec, arr)); results[(c, s)] = (rec, arr)
        td = rec["test_default"]
        log(f"[exp2] {i:>2}/{len(todo)} {display_name:<12} {c:<13} seed={s} | {rec['fit_seconds']:5.0f}s best_ep={out['best_epoch']} | val PR-AUC={rec['cv_pooled_ap']*100:5.1f} | "
            f"test@0.5: F1f={td['f1_fake']*100:5.1f} Rf={td['recall_fake']*100:5.1f} Pf={td['precision_fake']*100:5.1f} PR-AUC={td['pr_auc']*100:5.1f} | elapsed {fmt_time(time.time() - t_start)}")
    return results

BERT_RES = {}
if BERT_AVAILABLE and RUN["exp2_bert"]:
    BERT_RES[("BanglaBERT", "clean")] = run_exp2()
    for mid in EXTRA_TRANSFORMERS:                                   # optional additional checkpoints (off by default)
        BERT_RES[(mid.split("/")[-1], "clean")] = run_exp2(mid, mid.split("/")[-1])
    if P["bert_input_ablation"]:                                     # BanglaBERT on the raw (un-stripped) normalised headline
        BERT_RES[("BanglaBERT", "raw")] = run_exp2(variant="raw", conditions=["none"])
print({k: len(v) for k, v in BERT_RES.items()})

In [ ]:
TRANSFORMER = pd.DataFrame([flatten(rec, family="transformer") for res in BERT_RES.values() for (rec, _) in res.values()])
if len(TRANSFORMER):
    as_pct(TRANSFORMER).to_csv(OUT / "tables" / "exp2_transformer_all_runs.csv", index=False)
    hist = pd.DataFrame([dict(model=rec["model"], condition=rec["condition"], seed=rec["seed"], **h) for res in BERT_RES.values() for (rec, _) in res.values() for h in rec["history"]])
    hist.to_csv(OUT / "tables" / "exp2_transformer_training_history.csv", index=False)
    agg = (TRANSFORMER.groupby(["model", "condition"])[["d_f1_fake", "d_recall_fake", "d_precision_fake", "d_macro_f1", "d_mcc", "pr_auc", "roc_auc", "t_f1_fake"]]
           .agg(["mean", "std"]).mul(100).round(2))
    show(agg)
    # training curves (validation PR-AUC and loss) for the 'none' condition
    fig, axs = plt.subplots(1, 2, figsize=(10, 3.4))
    for (mname, cnd, sd), g in hist.groupby(["model", "condition", "seed"]):
        if mname != "BanglaBERT" or cnd not in ("none", "class_weight"): continue
        col = PALETTE[0] if cnd == "none" else PALETTE[3]
        axs[0].plot(g["epoch"], g["train_loss"], color=col, alpha=.6, label=f"{cnd} (train)" if sd == hist.seed.min() else None)
        axs[0].plot(g["epoch"], g["val_loss"], color=col, ls="--", alpha=.6, label=f"{cnd} (val)" if sd == hist.seed.min() else None)
        axs[1].plot(g["epoch"], g["val_pr_auc"] * 100, color=col, marker="o", alpha=.7, label=cnd if sd == hist.seed.min() else None)
    axs[0].set_title("BanglaBERT: loss per epoch"); axs[0].set_xlabel("Epoch"); axs[0].legend(fontsize=8)
    axs[1].set_title("BanglaBERT: validation PR-AUC (fake)"); axs[1].set_xlabel("Epoch"); axs[1].set_ylabel("%"); axs[1].legend(fontsize=8)
    fig.tight_layout(); savefig(fig, "fig04_banglabert_training_curves")

## 10. Master results, confidence intervals, justified model selection
*All numbers in this section come from the **test split**, scored once; all **selection** decisions use only training-side evidence
(inner-CV out-of-fold scores for classical models, the 10 % validation split for transformers).*

**Selection criteria** (guide §5 — "the preferable model depends on whether missed fake news or false alarms carry greater cost"):

| ID | Criterion (maximise / minimise on CV or validation evidence) | Operating point reported on the test set |
|---|---|---|
| S1 | PR-AUC of the fake class (threshold-free ranking quality) | F1-optimal threshold (from training-side scores) |
| S2 | fake-class F1 at the F1-optimal threshold | F1-optimal threshold |
| S3 | fake recall subject to fake precision ≥ `precision_floor` (default 0.50) | the threshold achieving that policy |
| S4(r) | expected cost per article `(r·FN + FP)/N` for cost ratios r ∈ {1, 2, 5, 10, 20} | the cost-minimising threshold |

In [ ]:
ALL = pd.concat([CLASSICAL, TRANSFORMER], ignore_index=True) if len(TRANSFORMER) else CLASSICAL.copy()
assert len(ALL), "No results available - enable at least one experiment in the RUN switches."
NUM = [c for c in ALL.select_dtypes(include=[np.number]).columns if c != "seed"]
KEYS = ["family", "model", "condition"]
AGG = ALL.groupby(KEYS, sort=False)[NUM].mean().reset_index()
STD = ALL.groupby(KEYS, sort=False)[NUM].std().reset_index()
NSEED = ALL.groupby(KEYS, sort=False).size().rename("n_seeds").reset_index()
AGG = AGG.merge(NSEED, on=KEYS)
MODEL_ORDER = [m for m in MODELS if m in set(AGG.model)] + [m for m in AGG.model.unique() if m not in MODELS]
AGG["_mo"] = AGG.model.map({m: i for i, m in enumerate(MODEL_ORDER)}); AGG["_co"] = AGG.condition.map({c: i for i, c in enumerate(CONDITIONS)})
AGG = AGG.sort_values(["_mo", "_co"]).drop(columns=["_mo", "_co"]).reset_index(drop=True)

# representative prediction arrays per configuration (classical: the single run; transformers: the first seed, fixed in advance)
REP = {}
for (m, c), (rec, arr) in EXP1.items(): REP[(m, c)] = (rec, arr)
for res in BERT_RES.values():
    for (c, s), (rec, arr) in sorted(res.items(), key=lambda kv: kv[0][1]):
        REP.setdefault((rec["model"], c), (rec, arr))
print(f"{len(AGG)} configurations in the master table; {sum(1 for _ in REP)} with stored predictions")

def point_threshold(rec, point):
    if point == "default": return rec["thr_default"]
    if point == "tuned": return rec["thr_f1"]
    if point == "policy": return rec["thr_policy"]
    if isinstance(point, tuple) and point[0] == "cost": return rec["cost"][point[1]]["thr"]
    raise ValueError(point)

def preds_at(model, cond, point):
    rec, arr = REP[(model, cond)]
    thr = point_threshold(rec, point)
    return (arr["test_score"] >= thr).astype(int) if np.isfinite(thr) else None

# ---- bootstrap CIs for every configuration (threshold-based metrics) ----
def ytest_of(model, cond): return REP[(model, cond)][1]["test_y"]          # labels stored with every result (smoke BERT may use a test subset)
CI_KEYS = ("precision_fake", "recall_fake", "f1_fake", "macro_f1", "mcc", "balanced_acc")
ci_rows = []
for (m, c), (rec, arr) in REP.items():
    yv = ytest_of(m, c)
    for point, tag in [("default", "d"), ("tuned", "t")]:
        pr = preds_at(m, c, point); ci = boot_ci_threshold_metrics(yv, pr, B=P["n_boot"], seed=SEED, keys=CI_KEYS)
        row = dict(model=m, condition=c, point=tag)
        for k, (lo, hi) in ci.items(): row[f"{k}_lo"] = lo; row[f"{k}_hi"] = hi
        ci_rows.append(row)
CI = pd.DataFrame(ci_rows)

# AUC CIs only for the configurations with the best CV/validation PR-AUC (bootstrapping AUCs is slower)
top_cfg = AGG.sort_values("cv_pooled_pr_auc", ascending=False).head(P["ci_auc_topk"])[["model", "condition"]].itertuples(index=False)
auc_rows = []
for m, c in top_cfg:
    rec, arr = REP[(m, c)]; ci = boot_ci_auc(arr["test_y"], arr["test_score"], B=min(P["n_boot"], 500), seed=SEED)
    auc_rows.append(dict(model=m, condition=c, pr_auc_lo=ci["pr_auc"][0], pr_auc_hi=ci["pr_auc"][1], roc_auc_lo=ci["roc_auc"][0], roc_auc_hi=ci["roc_auc"][1]))
AUCCI = pd.DataFrame(auc_rows)
CI.to_csv(OUT / "tables" / "bootstrap_ci_threshold_metrics.csv", index=False); AUCCI.to_csv(OUT / "tables" / "bootstrap_ci_auc_top_configs.csv", index=False)

In [ ]:
# ---- publication tables: "value [95% CI]" ----
def cell(v, lo=None, hi=None, nd=1):
    if pd.isna(v): return "n/a"
    return f"{v * 100:.{nd}f}" + (f" [{lo * 100:.{nd}f}, {hi * 100:.{nd}f}]" if lo is not None and not pd.isna(lo) else "")

def build_main_table(point):
    tag = "d" if point == "default" else "t"
    rows = []
    for _, r in AGG.iterrows():
        ci = CI[(CI.model == r.model) & (CI.condition == r.condition) & (CI.point == tag)]
        ci = ci.iloc[0] if len(ci) else None
        g = lambda k: cell(r[f"{tag}_{k}"], None if ci is None else ci[f"{k}_lo"], None if ci is None else ci[f"{k}_hi"])
        a = AUCCI[(AUCCI.model == r.model) & (AUCCI.condition == r.condition)]
        rows.append({"Model": r.model, "Condition": r.condition, "n runs": int(r.n_seeds),
                     "Fake P": g("precision_fake"), "Fake R": g("recall_fake"), "Fake F1": g("f1_fake"), "Macro-F1": g("macro_f1"),
                     "Weighted-F1": cell(r[f"{tag}_weighted_f1"]), "Bal. Acc": g("balanced_acc"), "MCC": g("mcc"), "Accuracy": cell(r[f"{tag}_accuracy"]),
                     "PR-AUC": cell(r.pr_auc, a.pr_auc_lo.iloc[0] if len(a) else None, a.pr_auc_hi.iloc[0] if len(a) else None),
                     "ROC-AUC": cell(r.roc_auc, a.roc_auc_lo.iloc[0] if len(a) else None, a.roc_auc_hi.iloc[0] if len(a) else None)})
    return pd.DataFrame(rows)

def to_latex_safe(d, path, caption, label):
    try:
        (OUT / "tables" / path).write_text(d.to_latex(index=False, escape=True, caption=caption, label=label), encoding="utf-8")
    except Exception as e:
        print("LaTeX export skipped:", repr(e))

T_DEF = build_main_table("default"); T_TUN = build_main_table("tuned")
T_DEF.to_csv(OUT / "tables" / "main_results_default_threshold.csv", index=False); T_TUN.to_csv(OUT / "tables" / "main_results_tuned_threshold.csv", index=False)
to_latex_safe(T_DEF, "main_results_default_threshold.tex", "Test-set results at the default decision threshold (value [95\\% bootstrap CI]; transformers: mean over seeds, CI from the first seed).", "tab:main_default")
to_latex_safe(T_TUN, "main_results_tuned_threshold.tex", "Test-set results at the threshold tuned on training-side scores (F1-optimal).", "tab:main_tuned")
print("MAIN TABLE - default decision threshold (positive class = fake)"); show(T_DEF)

# numeric tables (percent) for later analysis
as_pct(AGG).to_csv(OUT / "tables" / "master_numeric_aggregated.csv", index=False)
if len(TRANSFORMER) and (TRANSFORMER.groupby(["model", "condition"]).size() > 1).any():
    sd = STD[STD.family == "transformer"][KEYS + ["d_f1_fake", "d_recall_fake", "d_precision_fake", "d_macro_f1", "d_mcc", "pr_auc"]]
    as_pct(sd).to_csv(OUT / "tables" / "transformer_seed_std.csv", index=False)

In [ ]:
# ---- CV-/validation-based selection of finalists (never the test set) ----
def best_row(d, col, maximise=True):
    d = d.dropna(subset=[col])
    if not len(d): return None
    return d.loc[d[col].idxmax() if maximise else d[col].idxmin()]

SEL_ROWS = []
criteria = [("S1: CV PR-AUC", "cv_pooled_pr_auc", True, "tuned"), ("S2: CV F1-fake @tuned thr", "cvt_f1_fake", True, "tuned"),
            ("S3: CV recall @ precision>=floor", "cvp_recall_fake", True, "policy")]
criteria += [(f"S4: min CV cost (FN:FP={r}:1)", f"oofcost_r{r}", False, ("cost", r)) for r in CONFIG["cost_ratios"]]
for fam in ["classical", "transformer"]:
    sub = AGG[AGG.family == fam]
    if not len(sub): continue
    for name, col, mx, point in criteria:
        b = best_row(sub, col, mx)
        if b is None: continue
        rec, arr = REP[(b.model, b.condition)]; yv = arr["test_y"]
        thr = point_threshold(rec, point); pred = (arr["test_score"] >= thr).astype(int)
        tm = threshold_metrics(yv, arr["test_score"], thr); ci = boot_ci_threshold_metrics(yv, pred, B=P["n_boot"], seed=SEED, keys=("f1_fake", "recall_fake", "precision_fake"))
        rk = ranking_metrics(yv, arr["test_score"])
        SEL_ROWS.append(dict(family=fam, criterion=name, model=b.model, condition=b.condition,
                             selection_value=round(float(b[col]) * (100 if "cost" not in col else 1), 3), operating_point=str(point), threshold=round(float(thr), 4),
                             test_precision_fake=round(tm["precision_fake"] * 100, 2), test_recall_fake=round(tm["recall_fake"] * 100, 2), test_f1_fake=round(tm["f1_fake"] * 100, 2),
                             test_f1_ci=f"[{ci['f1_fake'][0] * 100:.1f}, {ci['f1_fake'][1] * 100:.1f}]", test_macro_f1=round(tm["macro_f1"] * 100, 2), test_mcc=round(tm["mcc"] * 100, 2),
                             test_pr_auc=round(rk["pr_auc"] * 100, 2), test_roc_auc=round(rk["roc_auc"] * 100, 2), test_tp=tm["tp"], test_fp=tm["fp"], test_fn=tm["fn"], test_tn=tm["tn"]))
SEL = pd.DataFrame(SEL_ROWS)
SEL.to_csv(OUT / "tables" / "selection_summary_finalists.csv", index=False)
print("FINALISTS chosen on CV / validation evidence, reported on the test set"); show(SEL[["family", "criterion", "model", "condition", "selection_value", "test_precision_fake", "test_recall_fake", "test_f1_fake", "test_f1_ci", "test_macro_f1", "test_mcc"]])

In [ ]:
# ---- head-to-head: best classical finalist vs best BanglaBERT finalist (same test cases: McNemar + paired bootstrap) ----
H2H = []
if len(SEL) and (SEL.family == "transformer").any():
    for crit in SEL.criterion.unique():
        a = SEL[(SEL.criterion == crit) & (SEL.family == "classical")]; b = SEL[(SEL.criterion == crit) & (SEL.family == "transformer")]
        if not len(a) or not len(b): continue
        a, b = a.iloc[0], b.iloc[0]
        point_of = {c[0]: c[3] for c in criteria}[crit]
        pa, pb = preds_at(a.model, a.condition, point_of), preds_at(b.model, b.condition, point_of)
        if pa is None or pb is None or len(pa) != len(pb): continue
        yv = REP[(a.model, a.condition)][1]["test_y"]
        if len(yv) != len(REP[(b.model, b.condition)][1]["test_y"]): continue
        bb, cc, pm = mcnemar_exact(yv, pa, pb)
        dm, lo, hi, pbt = paired_bootstrap_diff(yv, pa, pb, "f1_fake", B=P["n_boot"], seed=SEED)
        H2H.append(dict(criterion=crit, classical=f"{a.model} / {a.condition}", transformer=f"{b.model} / {b.condition}", f1_fake_classical=a.test_f1_fake, f1_fake_transformer=b.test_f1_fake,
                        diff_f1_pp=round(dm * 100, 2), diff_ci_lo_pp=round(lo * 100, 2), diff_ci_hi_pp=round(hi * 100, 2), paired_bootstrap_p=round(pbt, 4),
                        mcnemar_b_classical_only_correct=bb, mcnemar_c_transformer_only_correct=cc, mcnemar_p=round(pm, 4)))
H2H = pd.DataFrame(H2H)
if len(H2H):
    H2H.to_csv(OUT / "tables" / "head_to_head_classical_vs_transformer.csv", index=False)
    print("HEAD-TO-HEAD (diff = classical minus transformer, percentage points of fake-F1)"); show(H2H)

### 10b. Figures for the main results
`fig05` heat-maps (ranking quality vs operating point) · `fig06` precision/recall by condition · `fig07/08` PR and ROC curves of the
finalists · `fig09` operating-point scatter with iso-F1 curves · `fig10` confusion matrices · `fig11` flag-rate trade-off ·
`fig12` cost-sensitive comparison. Every figure is saved as 300-dpi PNG **and** vector PDF.

In [ ]:
COND_LABEL = {"none": "None", "class_weight": "Class wt.", "ros": "ROS", "rus": "RUS", "smote": "SMOTE", "text_aug": "Text-aug"}
COND_COLOR = {c: PALETTE[i] for i, c in enumerate(CONDITIONS)}
COND_MARK = {"none": "o", "class_weight": "s", "ros": "^", "rus": "v", "smote": "D", "text_aug": "P"}
MODEL_COLOR = {m: (PALETTE + ["#8C564B", "#17BECF", "#BCBD22", "#E377C2"])[i % 13] for i, m in enumerate(MODEL_ORDER)}

def heat(ax, col, title, scale=100, fmt="{:.1f}", cmap="viridis"):
    M = np.full((len(MODEL_ORDER), len(CONDITIONS)), np.nan)
    for i, m in enumerate(MODEL_ORDER):
        for j, c in enumerate(CONDITIONS):
            r = AGG[(AGG.model == m) & (AGG.condition == c)]
            if len(r) and col in r.columns: M[i, j] = r.iloc[0][col] * scale
    cm = plt.get_cmap(cmap).copy(); cm.set_bad("#DDDDDD")
    im = ax.imshow(np.ma.masked_invalid(M), cmap=cm, aspect="auto")
    lo, hi = np.nanmin(M), np.nanmax(M)
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            if np.isnan(M[i, j]): ax.text(j, i, "n/a", ha="center", va="center", fontsize=7, color="#555555")
            else: ax.text(j, i, fmt.format(M[i, j]), ha="center", va="center", fontsize=7.5, color="white" if (M[i, j] - lo) / (hi - lo + 1e-9) < .55 else "black")
    ax.set_xticks(range(len(CONDITIONS))); ax.set_xticklabels([COND_LABEL[c] for c in CONDITIONS], rotation=30, ha="right")
    ax.set_yticks(range(len(MODEL_ORDER))); ax.set_yticklabels(MODEL_ORDER); ax.set_title(title, fontsize=10); ax.grid(False)
    plt.colorbar(im, ax=ax, fraction=.04, pad=.02)

fig, axs = plt.subplots(2, 2, figsize=(13, 9.5))
heat(axs[0, 0], "pr_auc", "(a) PR-AUC of fake class (%) — ranking quality")
heat(axs[0, 1], "d_recall_fake", "(b) Fake recall (%) at default threshold")
heat(axs[1, 0], "d_f1_fake", "(c) Fake F1 (%) at default threshold")
heat(axs[1, 1], "t_f1_fake", "(d) Fake F1 (%) at tuned threshold")
fig.suptitle("Effect of imbalance mitigation on test-set performance (positive class = fake)", y=1.0, fontsize=12)
fig.tight_layout(); savefig(fig, "fig05_heatmaps_model_by_condition")

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(13, 7.2), sharex=True)
xs = np.arange(len(MODEL_ORDER)); w = 0.85 / len(CONDITIONS)
for ax, col, ttl in [(axs[0], "d_recall_fake", "Fake-class recall (%) at the default threshold"), (axs[1], "d_precision_fake", "Fake-class precision (%) at the default threshold")]:
    for j, c in enumerate(CONDITIONS):
        v = [AGG[(AGG.model == m) & (AGG.condition == c)][col].mean() * 100 if len(AGG[(AGG.model == m) & (AGG.condition == c)]) else np.nan for m in MODEL_ORDER]
        ax.bar(xs + j * w, v, w, label=COND_LABEL[c], color=COND_COLOR[c], edgecolor="black", linewidth=.4)
    ax.set_ylabel("%"); ax.set_title(ttl); ax.set_ylim(0, 105)
axs[0].legend(ncol=6, fontsize=8, loc="upper center", bbox_to_anchor=(.5, 1.28))
axs[1].set_xticks(xs + .4); axs[1].set_xticklabels(MODEL_ORDER, rotation=20, ha="right")
fig.tight_layout(); savefig(fig, "fig06_recall_precision_by_condition")

In [ ]:
# ---- finalists to draw: CV-selected winners + the two reference points of the ML paper ----
def finalist_list():
    out = []
    if len(SEL):
        for fam in ["classical", "transformer"]:
            for crit in ["S1: CV PR-AUC", "S2: CV F1-fake @tuned thr", "S3: CV recall @ precision>=floor"]:
                r = SEL[(SEL.family == fam) & (SEL.criterion == crit)]
                if len(r): out.append((r.iloc[0].model, r.iloc[0].condition))
    for ref in [("Naive Bayes", "ros"), ("Logistic Regression", "none")]:
        if ref in REP: out.append(ref)
    seen, res = set(), []
    for x in out:
        if x not in seen and x in REP: seen.add(x); res.append(x)
    return res[:8]
FINALS = finalist_list(); print("curves drawn for:", FINALS)

fig, axs = plt.subplots(1, 2, figsize=(12.5, 5))
for k, (m, c) in enumerate(FINALS):
    rec, arr = REP[(m, c)]; yv, s = arr["test_y"], arr["test_score"]
    p, r, _ = precision_recall_curve(yv, s); fpr, tpr, _ = roc_curve(yv, s); rk = ranking_metrics(yv, s); col = PALETTE[k % len(PALETTE)]
    axs[0].plot(r, p, color=col, lw=1.8, label=f"{m} / {COND_LABEL[c]} (AP={rk['pr_auc'] * 100:.1f})")
    tm = threshold_metrics(yv, s, rec["thr_default"]); axs[0].scatter([tm["recall_fake"]], [tm["precision_fake"]], color=col, marker="o", s=45, zorder=5, edgecolor="black")
    axs[1].plot(fpr, tpr, color=col, lw=1.8, label=f"{m} / {COND_LABEL[c]} (AUC={rk['roc_auc'] * 100:.1f})")
prev = float(np.mean(REP[FINALS[0]][1]["test_y"])) if FINALS else 0.15
axs[0].axhline(prev, color="grey", ls=":", lw=1, label=f"No-skill (prevalence {prev * 100:.1f}%)"); axs[1].plot([0, 1], [0, 1], color="grey", ls=":", lw=1)
axs[0].set_xlabel("Recall (fake)"); axs[0].set_ylabel("Precision (fake)"); axs[0].set_title("Precision–recall curves (dots = default threshold)"); axs[0].legend(fontsize=7, loc="upper right")
axs[1].set_xlabel("False-positive rate"); axs[1].set_ylabel("True-positive rate (recall)"); axs[1].set_title("ROC curves"); axs[1].legend(fontsize=7, loc="lower right")
fig.tight_layout(); savefig(fig, "fig07_08_pr_and_roc_curves_finalists")

In [ ]:
# ---- operating-point scatter at the default threshold, with iso-F1 curves ----
fig, ax = plt.subplots(figsize=(8.2, 6.2))
rr = np.linspace(0.01, 1, 200)
for f in (0.4, 0.5, 0.6, 0.7, 0.8):
    pp = f * rr / (2 * rr - f); ok = (pp > 0) & (pp <= 1); ax.plot(rr[ok], pp[ok], color="lightgrey", lw=1, zorder=0); ax.text(rr[ok][-1] if ok.any() else .98, pp[ok][-1] if ok.any() else .5, f"F1={f}", fontsize=7, color="grey")
for _, r in AGG.iterrows():
    ax.scatter(r.d_recall_fake, r.d_precision_fake, marker=COND_MARK[r.condition], color=MODEL_COLOR[r.model], s=55, edgecolor="black", linewidth=.5, alpha=.9)
ax.set_xlabel("Fake recall (default threshold)"); ax.set_ylabel("Fake precision (default threshold)"); ax.set_xlim(0, 1.02); ax.set_ylim(0, 1.02)
ax.set_title("Operating points of all configurations (test set)")
from matplotlib.lines import Line2D
h1 = [Line2D([0], [0], marker="o", ls="", color=MODEL_COLOR[m], label=m, markeredgecolor="black") for m in MODEL_ORDER]
h2 = [Line2D([0], [0], marker=COND_MARK[c], ls="", color="#444444", label=COND_LABEL[c]) for c in CONDITIONS]
fig.subplots_adjust(right=0.80)
fig.legend(handles=h1, fontsize=7, loc="upper left", bbox_to_anchor=(0.81, 0.88), title="Model", title_fontsize=8)
fig.legend(handles=h2, fontsize=7, loc="lower left", bbox_to_anchor=(0.81, 0.12), title="Condition", title_fontsize=8)
savefig(fig, "fig09_operating_points_iso_f1")

In [ ]:
# ---- confusion matrices of key configurations ----
cm_list = []
for fam in ["classical", "transformer"]:
    r = SEL[(SEL.family == fam) & (SEL.criterion == "S2: CV F1-fake @tuned thr")] if len(SEL) else []
    if len(r): cm_list.append((r.iloc[0].model, r.iloc[0].condition, "tuned", f"{fam} finalist (S2)"))
r3 = SEL[(SEL.family == "classical") & (SEL.criterion == "S3: CV recall @ precision>=floor")] if len(SEL) else []
if len(r3): cm_list.append((r3.iloc[0].model, r3.iloc[0].condition, "policy", "classical, recall@precision floor (S3)"))
if ("Naive Bayes", "ros") in REP: cm_list.append(("Naive Bayes", "ros", "default", "ML-paper winner (default thr.)"))
cm_list = cm_list[:4]
fig, axs = plt.subplots(1, len(cm_list), figsize=(3.7 * len(cm_list), 3.6)); axs = np.atleast_1d(axs)
for ax, (m, c, pt, ttl) in zip(axs, cm_list):
    rec, arr = REP[(m, c)]; yv = arr["test_y"]; thr = point_threshold(rec, pt); pr_ = (arr["test_score"] >= thr).astype(int)
    cmx = confusion_matrix(yv, pr_, labels=[1, 0]); nrm = cmx / cmx.sum(1, keepdims=True)
    ax.imshow(nrm, cmap="Blues", vmin=0, vmax=1); ax.grid(False)
    for i in range(2):
        for j in range(2): ax.text(j, i, f"{cmx[i, j]}\n({nrm[i, j] * 100:.1f}%)", ha="center", va="center", color="white" if nrm[i, j] > .5 else "black", fontsize=9)
    ax.set_xticks([0, 1]); ax.set_xticklabels(["Pred. fake", "Pred. authentic"], fontsize=8); ax.set_yticks([0, 1]); ax.set_yticklabels(["Actual fake", "Actual authentic"], fontsize=8)
    ax.set_title(f"{m} / {COND_LABEL[c]}\n{ttl}", fontsize=8.5)
fig.tight_layout(); savefig(fig, "fig10_confusion_matrices")

In [ ]:
# ---- trade-off between alert volume and fake recall / precision for the finalists ----
fig, axs = plt.subplots(1, 2, figsize=(11.5, 4))
for k, (m, c) in enumerate(FINALS[:5]):
    rec, arr = REP[(m, c)]; yv, s = arr["test_y"], arr["test_score"]; order = np.argsort(-s); ys = yv[order]
    tp = np.cumsum(ys); n = np.arange(1, len(ys) + 1); col = PALETTE[k % len(PALETTE)]
    axs[0].plot(n / len(ys) * 100, tp / ys.sum() * 100, color=col, label=f"{m} / {COND_LABEL[c]}"); axs[1].plot(n / len(ys) * 100, tp / n * 100, color=col)
    for point, mk in [("default", "o"), ("tuned", "*")]:
        tm = threshold_metrics(yv, s, point_threshold(rec, point)); axs[0].scatter(tm["flag_rate"] * 100, tm["recall_fake"] * 100, color=col, marker=mk, s=60, edgecolor="black", zorder=5)
        axs[1].scatter(tm["flag_rate"] * 100, tm["precision_fake"] * 100, color=col, marker=mk, s=60, edgecolor="black", zorder=5)
axs[0].set_xlabel("% of articles flagged (highest scores first)"); axs[0].set_ylabel("Fake recall (%)"); axs[0].set_title("Recall vs. review workload"); axs[0].legend(fontsize=7)
axs[1].set_xlabel("% of articles flagged"); axs[1].set_ylabel("Fake precision (%)"); axs[1].set_title("Precision vs. review workload (● default thr., ★ tuned thr.)"); axs[1].set_ylim(0, 105)
fig.tight_layout(); savefig(fig, "fig11_alert_volume_tradeoff")

In [ ]:
# ---- cost-sensitive comparison: expected cost per article for different FN:FP cost ratios ----
ratios = CONFIG["cost_ratios"]; prevalence = float(y_te.mean())
fig, ax = plt.subplots(figsize=(7.4, 4.4))
ax.plot(ratios, [r * prevalence for r in ratios], color="grey", ls=":", label="Flag nothing (all authentic)")
ax.plot(ratios, [(1 - prevalence)] * len(ratios), color="grey", ls="--", label="Flag everything")
if ("Naive Bayes", "ros") in REP:
    ax.plot(ratios, [REP[("Naive Bayes", "ros")][0]["cost"][r]["test_default_cost"] for r in ratios], color=PALETTE[2], marker="s", label="ML-paper winner (NB + ROS, default thr.)")
for fam, col in [("classical", PALETTE[0]), ("transformer", PALETTE[3])]:
    ys_, labs = [], []
    for r in ratios:
        s_ = SEL[(SEL.family == fam) & (SEL.criterion == f"S4: min CV cost (FN:FP={r}:1)")] if len(SEL) else []
        if len(s_): ys_.append(REP[(s_.iloc[0].model, s_.iloc[0].condition)][0]["cost"][r]["test_cost"]); labs.append(f"{s_.iloc[0].model}/{COND_LABEL[s_.iloc[0].condition]}")
        else: ys_.append(np.nan); labs.append("")
    if np.isfinite(ys_).any(): ax.plot(ratios, ys_, color=col, marker="o", label=f"CV-selected {fam} (cost-optimal threshold)")
ax.set_xscale("log"); ax.set_xticks(ratios); ax.set_xticklabels([f"{r}:1" for r in ratios]); ax.set_xlabel("Cost of a missed fake : cost of a false alarm"); ax.set_ylabel("Expected cost per article (test)")
ax.set_title("Cost-sensitive comparison"); ax.legend(fontsize=7.5)
fig.tight_layout(); savefig(fig, "fig12_cost_sensitive_comparison")

## 11. Experiment 3 — Stability and statistical significance (repeated stratified k‑fold)
A single 80/20 split cannot support claims such as "oversampling is best for five of seven models" (a limitation the ML paper itself noted).
Here **all** configurations are re-evaluated with *repeated stratified k‑fold CV* over the whole analysed dataset, using the **default
(un-tuned) hyper-parameters** so that no test fold was ever used for tuning (the tuned results of §8 stay the primary evidence).

* confidence intervals use the Nadeau–Bengio variance correction for overlapping training sets;
* *condition vs. no mitigation* is tested per model with the corrected resampled t‑test (Holm-adjusted across comparisons);
* all configurations are ranked with the Friedman test (folds are blocks) and average ranks.

In [ ]:
X_all = df["clean_text"].values; y_all = df["is_fake"].values.astype(int)

def cv_eval_config(model, cond, splits, X, y, feat="word_paper"):
    def one(tr, te):
        pipe = build_pipeline(model, cond, feat=feat); pipe.fit(X[tr], y[tr]); s = get_score(pipe, X[te], model)
        m = full_metrics(y[te], s, default_threshold(model)); m.update(n_train=int(len(tr)), n_test=int(len(te)), n_fake_test=int(y[te].sum())); return m
    return Parallel(n_jobs=N_JOBS)(      # one constant pool size: changing n_jobs between calls forces joblib to re-spawn all workers
        delayed(one)(tr, te) for tr, te in splits)

def run_cv_grid(tag, models, conds, splits, X, y, feat="word_paper"):
    rows, t0 = [], time.time(); todo = [(m, c) for m in models for c in conds if applicable(m, c)]
    for i, (m, c) in enumerate(todo, 1):
        key = f"{tag}__{m}__{c}__{feat}__{len(splits)}"
        if cache_has(key): res = cache_get(key)
        else:
            res = cv_eval_config(m, c, splits, X, y, feat); cache_put(key, res)
            log(f"[{tag}] {i:>2}/{len(todo)} {m:<20} {c:<13} {feat:<22} F1f={np.nanmean([r['f1_fake'] for r in res]) * 100:5.1f} PR-AUC={np.nanmean([r['pr_auc'] for r in res]) * 100:5.1f} | elapsed {fmt_time(time.time() - t0)}")
        for k, r in enumerate(res): rows.append(dict(model=m, condition=c, feat=feat, fold=k, **r))
    return pd.DataFrame(rows)

CV_METRICS = ["f1_fake", "recall_fake", "precision_fake", "macro_f1", "mcc", "balanced_acc", "pr_auc", "roc_auc"]
def summarise_cv(d, extra_keys=()):
    out = []
    for keys, g in d.groupby(["model", "condition", *extra_keys], sort=False):
        J = len(g); corr = 1.0 / J + g.n_test.iloc[0] / g.n_train.iloc[0]; tcrit = stats.t.ppf(0.975, max(J - 1, 1)); row = dict(zip(["model", "condition", *extra_keys], keys)); row["n_folds"] = J
        for k in CV_METRICS:
            v = g[k].values.astype(float); mu = np.nanmean(v); sd = np.nanstd(v, ddof=1) if J > 1 else np.nan; se = np.sqrt(corr * np.nanvar(v, ddof=1)) if J > 1 else np.nan
            row[f"{k}_mean"] = mu; row[f"{k}_sd"] = sd; row[f"{k}_ci_lo"] = mu - tcrit * se; row[f"{k}_ci_hi"] = mu + tcrit * se
        out.append(row)
    return pd.DataFrame(out)

REPCV, REPCV_SUM = pd.DataFrame(), pd.DataFrame()
if RUN["exp3_repeated_cv"]:
    SPL3 = list(RepeatedStratifiedKFold(n_splits=P["rep_folds"], n_repeats=P["rep_repeats"], random_state=SEED).split(np.zeros(len(y_all)), y_all))
    log(f"[exp3] {P['rep_folds']}-fold x {P['rep_repeats']} repeats = {len(SPL3)} folds per configuration")
    REPCV = run_cv_grid("exp3", MODELS, CONDITIONS, SPL3, X_all, y_all)
    REPCV.to_csv(OUT / "tables" / "exp3_repeated_cv_fold_level.csv", index=False)
    REPCV_SUM = summarise_cv(REPCV)
    REPCV_SUM.to_csv(OUT / "tables" / "exp3_repeated_cv_summary.csv", index=False)
    pretty = REPCV_SUM[["model", "condition", "n_folds"]].copy()
    for k in ["f1_fake", "recall_fake", "precision_fake", "macro_f1", "mcc", "pr_auc"]:
        pretty[k] = [f"{m * 100:.1f} ± {s * 100:.1f}" for m, s in zip(REPCV_SUM[f"{k}_mean"], REPCV_SUM[f"{k}_sd"])]
    pretty.to_csv(OUT / "tables" / "exp3_repeated_cv_mean_pm_sd.csv", index=False); show(pretty.head(12))

In [ ]:
def condition_vs_none_tests(d, metrics=("f1_fake", "macro_f1", "pr_auc", "recall_fake")):
    rows = []
    for met in metrics:
        sub = []
        for m, g in d.groupby("model", sort=False):
            base = g[g.condition == "none"].sort_values("fold")
            if not len(base): continue
            for c, gc in g.groupby("condition", sort=False):
                if c == "none": continue
                gc = gc.sort_values("fold"); dd = gc[met].values - base[met].values
                md, t, p = nb_corrected_ttest(dd[~np.isnan(dd)], base.n_train.iloc[0], base.n_test.iloc[0])
                sub.append(dict(metric=met, model=m, condition=c, mean_diff_pp=md * 100, t=t, p=p, wilcoxon_p=(stats.wilcoxon(dd).pvalue if np.any(dd != 0) else 1.0)))
        if sub:
            s = pd.DataFrame(sub); s["p_holm"] = holm(np.nan_to_num(s.p.values, nan=1.0)); rows.append(s)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

if len(REPCV):
    CVT = condition_vs_none_tests(REPCV); CVT.round(4).to_csv(OUT / "tables" / "exp3_condition_vs_none_tests.csv", index=False)
    print("Does a mitigation condition significantly change performance relative to 'none'? (corrected resampled t-test, Holm-adjusted)")
    show(CVT[CVT.metric == "f1_fake"].round(4).sort_values("p_holm").head(15))
    # Friedman test + average ranks (folds are blocks)
    blocks = {}
    for (m, c), g in REPCV.groupby(["model", "condition"], sort=False): blocks[(m, c)] = g.sort_values("fold")
    FRI = []
    for met in ["f1_fake", "macro_f1", "pr_auc"]:
        mat = np.column_stack([b[met].values for b in blocks.values()]); ranks = np.apply_along_axis(lambda r: stats.rankdata(-r), 1, mat); chi, p = stats.friedmanchisquare(*mat.T)
        FRI.append(pd.DataFrame(dict(metric=met, model=[k[0] for k in blocks], condition=[k[1] for k in blocks], avg_rank=ranks.mean(0), friedman_chi2=chi, friedman_p=p)))
    FRI = pd.concat(FRI, ignore_index=True); FRI.round(4).to_csv(OUT / "tables" / "exp3_friedman_average_ranks.csv", index=False)
    print({met: float(FRI[FRI.metric == met].friedman_p.iloc[0]) for met in ["f1_fake", "macro_f1", "pr_auc"]}, "<- Friedman p-values")
    # best configuration by mean macro-F1 vs the rest
    best = REPCV_SUM.loc[REPCV_SUM.macro_f1_mean.idxmax()]; bb = blocks[(best.model, best.condition)]; rows = []
    for (m, c), g in blocks.items():
        if (m, c) == (best.model, best.condition): continue
        dd = bb.macro_f1.values - g.macro_f1.values; md, t, p = nb_corrected_ttest(dd, bb.n_train.iloc[0], bb.n_test.iloc[0]); rows.append(dict(vs_model=m, vs_condition=c, mean_diff_macroF1_pp=md * 100, t=t, p=p))
    BST = pd.DataFrame(rows); BST["p_holm"] = holm(np.nan_to_num(BST.p.values, nan=1.0)); BST.round(4).sort_values("p").to_csv(OUT / "tables" / "exp3_best_macroF1_config_vs_all.csv", index=False)
    print(f"Best mean macro-F1 in repeated CV: {best.model} / {best.condition}  ({best.macro_f1_mean * 100:.2f}%)")

    # figure: CI plot per model/condition + average-rank bars
    fig, axs = plt.subplots(1, 2, figsize=(14, 4.8), gridspec_kw={"width_ratios": [1.5, 1]})
    mo = [m for m in MODEL_ORDER if m in set(REPCV_SUM.model)]; w = 0.85 / len(CONDITIONS)
    for j, c in enumerate(CONDITIONS):
        sub = REPCV_SUM[REPCV_SUM.condition == c].set_index("model").reindex(mo)
        axs[0].errorbar(np.arange(len(mo)) + j * w, sub.f1_fake_mean * 100, yerr=[(sub.f1_fake_mean - sub.f1_fake_ci_lo) * 100, (sub.f1_fake_ci_hi - sub.f1_fake_mean) * 100], fmt=COND_MARK[c], color=COND_COLOR[c], ms=5, capsize=2, label=COND_LABEL[c])
    axs[0].set_xticks(np.arange(len(mo)) + .4); axs[0].set_xticklabels(mo, rotation=25, ha="right"); axs[0].set_ylim(0, 100); axs[0].set_ylabel("Fake F1 (%) — mean and 95% CI"); axs[0].set_title(f"Repeated {P['rep_folds']}-fold CV x{P['rep_repeats']} (default hyper-parameters)"); axs[0].legend(fontsize=7, ncol=3)
    rk = FRI[FRI.metric == "macro_f1"].sort_values("avg_rank").head(15)
    axs[1].barh([f"{m} / {COND_LABEL[c]}" for m, c in zip(rk.model, rk.condition)][::-1], rk.avg_rank.values[::-1], color=PALETTE[0], edgecolor="black", linewidth=.4)
    axs[1].set_xlabel("Average Friedman rank by macro-F1 (lower = better)"); axs[1].set_title("Top-15 configurations")
    fig.tight_layout(); savefig(fig, "fig13_repeated_cv_stability_and_ranks")

In [ ]:
# ---- optional: BanglaBERT inside the same folds (first repeat only; expensive) ----
BERT_CV = pd.DataFrame()
if BERT_AVAILABLE and P["run_bert_cv"] and RUN["exp3_repeated_cv"] and len(REPCV):
    rows = []; nsp = P["rep_folds"] * P["bert_cv_repeats"]; t0 = time.time()
    for ci, c in enumerate(P["bert_conditions"]):
        for k, (tr, te) in enumerate(SPL3[:nsp]):
            key = f"exp3bert__{c}__{k}__e{P['bert_epochs']}__L{CONFIG['bert']['max_len']}"
            if cache_has(key): m = cache_get(key)
            else:
                fi, vi = train_test_split(np.arange(len(tr)), test_size=CONFIG["bert"]["val_frac"], stratify=y_all[tr], random_state=SEED)
                Tf, Tv, Tt = bert_texts(df.iloc[tr[fi]]), bert_texts(df.iloc[tr[vi]]), bert_texts(df.iloc[te])
                out = finetune_transformer(BERT_MODEL_NAME, c, SEED, Tf, y_all[tr[fi]], Tv, y_all[tr[vi]], Tt, y_all[te], P["bert_epochs"], P["bert_bs"], P["bert_patience"], CONFIG["bert"]["max_len"])
                m = full_metrics(y_all[te], out["test_probs"], 0.5); m.update(n_train=int(len(tr)), n_test=int(len(te)), n_fake_test=int(y_all[te].sum())); cache_put(key, m)
                log(f"[exp3-bert] {c:<13} fold {k + 1}/{nsp} F1f={m['f1_fake'] * 100:.1f} PR-AUC={m['pr_auc'] * 100:.1f} | elapsed {fmt_time(time.time() - t0)}")
            rows.append(dict(model="BanglaBERT", condition=c, feat="transformer", fold=k, **m))
    BERT_CV = pd.DataFrame(rows); BERT_CV.to_csv(OUT / "tables" / "exp3_bert_cv_fold_level.csv", index=False)
    bsum = summarise_cv(BERT_CV); bsum.to_csv(OUT / "tables" / "exp3_bert_cv_summary.csv", index=False); show(bsum[["model", "condition", "n_folds", "f1_fake_mean", "recall_fake_mean", "precision_fake_mean", "pr_auc_mean"]].round(4))
    # paired comparison with the best classical configuration (same folds)
    bestc = REPCV_SUM.loc[REPCV_SUM.macro_f1_mean.idxmax()]; cmpb = []
    for c in P["bert_conditions"]:
        a = REPCV[(REPCV.model == bestc.model) & (REPCV.condition == bestc.condition) & (REPCV.fold < nsp)].sort_values("fold"); b = BERT_CV[BERT_CV.condition == c].sort_values("fold")
        for met in ["f1_fake", "macro_f1", "pr_auc"]:
            md, t, p = nb_corrected_ttest(b[met].values - a[met].values, a.n_train.iloc[0], a.n_test.iloc[0]); cmpb.append(dict(bert_condition=c, vs=f"{bestc.model}/{bestc.condition}", metric=met, mean_diff_bert_minus_classical_pp=md * 100, t=t, p=p))
    pd.DataFrame(cmpb).round(4).to_csv(OUT / "tables" / "exp3_bert_vs_best_classical_paired.csv", index=False)

## 12. Experiment 4 — Stress test: unseen news sources (domain-grouped CV)
BanFakeNews fake articles come from a handful of satire/clickbait sites. If a classifier mainly memorises *site style*, performance
should collapse when whole domains are held out. `StratifiedGroupKFold` (groups = `domain`) keeps every domain entirely in either training
or test folds. Same default hyper-parameters as §11; results are compared with random-split CV.

In [ ]:
DOM_CONDS = ["none", "class_weight", "ros"]
DOMAIN_RES, DOMAIN_SUM = pd.DataFrame(), pd.DataFrame()
if RUN["exp4_domain_cv"]:
    sgkf = StratifiedGroupKFold(n_splits=P["group_folds"], shuffle=True, random_state=SEED)
    SPL4 = [(tr, te) for tr, te in sgkf.split(np.zeros(len(y_all)), y_all, groups=df["domain"].values) if y_all[te].sum() > 0 and (y_all[te] == 0).sum() > 0]
    log(f"[exp4] {len(SPL4)} usable domain-grouped folds; test fake counts: {[int(y_all[te].sum()) for _, te in SPL4]}")
    DOMAIN_RES = run_cv_grid("exp4", MODELS, DOM_CONDS, SPL4, X_all, y_all)
    DOMAIN_RES.to_csv(OUT / "tables" / "exp4_domain_grouped_fold_level.csv", index=False)
    DOMAIN_SUM = summarise_cv(DOMAIN_RES); DOMAIN_SUM.to_csv(OUT / "tables" / "exp4_domain_grouped_summary.csv", index=False)
    if len(REPCV_SUM):
        cmp_ = DOMAIN_SUM.merge(REPCV_SUM, on=["model", "condition"], suffixes=("_domain", "_random"))
        cmp_["delta_f1_fake_pp"] = (cmp_.f1_fake_mean_domain - cmp_.f1_fake_mean_random) * 100; cmp_["delta_pr_auc_pp"] = (cmp_.pr_auc_mean_domain - cmp_.pr_auc_mean_random) * 100
        cmp_[["model", "condition", "f1_fake_mean_random", "f1_fake_mean_domain", "delta_f1_fake_pp", "pr_auc_mean_random", "pr_auc_mean_domain", "delta_pr_auc_pp"]].round(4).to_csv(OUT / "tables" / "exp4_domain_vs_random_cv.csv", index=False)
        fig, axs = plt.subplots(1, 2, figsize=(13, 4)); mo = [m for m in MODEL_ORDER if m in set(cmp_.model)]
        for ax, met, ttl in [(axs[0], "pr_auc", "PR-AUC (fake) — threshold-free"), (axs[1], "f1_fake", "Fake F1 at default threshold")]:
            sub = cmp_[cmp_.condition == "ros"].set_index("model").reindex(mo); x = np.arange(len(mo))
            ax.bar(x - .2, sub[f"{met}_mean_random"] * 100, .4, color=PALETTE[0], edgecolor="black", linewidth=.4, label="Random split CV"); ax.bar(x + .2, sub[f"{met}_mean_domain"] * 100, .4, color=PALETTE[3], edgecolor="black", linewidth=.4, label="Unseen-domain CV")
            ax.set_xticks(x); ax.set_xticklabels(mo, rotation=25, ha="right"); ax.set_title(ttl + " (ROS)"); ax.set_ylabel("%"); ax.set_ylim(0, max(sub[f"{met}_mean_random"].max(), sub[f"{met}_mean_domain"].max()) * 118); ax.legend(fontsize=8, loc="upper right")
        fig.tight_layout(); savefig(fig, "fig14_unseen_domain_stress_test")

## 13. Experiment 5 — Does the feature representation matter?
Word uni+bi-grams with the **default** scikit-learn tokenizer (what the ML paper actually used), the same with a **Bangla-aware** tokenizer
(primary setting), an uncapped vocabulary, and **character n-grams** (2–5, word-bounded), which are robust to Bangla morphology and spelling variation.

In [ ]:
FEATS = ["word_default_tokenizer", "word_paper", "word_full", "char_wb"]; FEAT_MODELS = [m for m in ["Logistic Regression", "SVM", "Naive Bayes", "Complement NB"] if m in MODELS]
FEAT_RES, FEAT_SUM = pd.DataFrame(), pd.DataFrame()
if RUN["exp5_features"]:
    SPL5 = list(StratifiedKFold(n_splits=max(3, min(5, P["rep_folds"])), shuffle=True, random_state=SEED).split(np.zeros(len(y_all)), y_all))
    FEAT_RES = pd.concat([run_cv_grid("exp5", FEAT_MODELS, ["none", "class_weight", "ros"], SPL5, X_all, y_all, feat=f) for f in FEATS], ignore_index=True)
    FEAT_RES.to_csv(OUT / "tables" / "exp5_feature_sensitivity_fold_level.csv", index=False)
    FEAT_SUM = summarise_cv(FEAT_RES, extra_keys=("feat",)); FEAT_SUM.to_csv(OUT / "tables" / "exp5_feature_sensitivity_summary.csv", index=False)
    fig, axs = plt.subplots(1, 2, figsize=(13, 4))
    for ax, met, ttl in [(axs[0], "pr_auc", "PR-AUC (%), no mitigation"), (axs[1], "f1_fake", "Fake F1 (%), random oversampling")]:
        cnd = "none" if met == "pr_auc" else "ros"; sub = FEAT_SUM[FEAT_SUM.condition == cnd]; x = np.arange(len(FEAT_MODELS)); w = .8 / len(FEATS)
        for j, f in enumerate(FEATS):
            s = sub[sub.feat == f].set_index("model").reindex(FEAT_MODELS); ax.bar(x + j * w, s[f"{met}_mean"] * 100, w, yerr=s[f"{met}_sd"] * 100, capsize=2, color=PALETTE[j], edgecolor="black", linewidth=.4, label=f.replace("_", " "))
        ax.set_xticks(x + .4 - w / 2); ax.set_xticklabels(FEAT_MODELS); ax.set_title(ttl); ax.set_ylabel("%")
    axs[0].legend(fontsize=7); fig.tight_layout(); savefig(fig, "fig15_feature_representation_sensitivity")

## 14. Error analysis, shortcut baselines and interpretability
Questions answered with *data*, not assumed: **which kind of fake news is missed? which authentic articles are falsely flagged?
how much of the performance could be obtained from shortcuts (headline length, publication source, a leaky metadata field)?
which n-grams does the linear model rely on?**
(CSV files containing Bangla text are written as UTF-8 with BOM so that Excel displays them correctly.)

In [ ]:
def meta_for(model, cond):
    rec, arr = REP[(model, cond)]
    return df_te if len(arr["test_y"]) == len(df_te) else df_te.iloc[B_TE]

def error_tables(model, cond, point="tuned", tag=""):
    rec, arr = REP[(model, cond)]; thr = point_threshold(rec, point); meta = meta_for(model, cond).reset_index(drop=True)
    d = meta.assign(score=arr["test_score"], pred=(arr["test_score"] >= thr).astype(int), y=arr["test_y"])
    fk = d[d.y == 1]; au = d[d.y == 0]
    t_type = fk.groupby("ftype").agg(n_fake=("y", "size"), recall=("pred", "mean")).reset_index(); t_type["recall"] = (t_type.recall * 100).round(1)
    t_cat = au.groupby("category").agg(n_authentic=("y", "size"), false_positive_rate=("pred", "mean")).reset_index().sort_values("n_authentic", ascending=False).head(10); t_cat["false_positive_rate"] = (t_cat.false_positive_rate * 100).round(1)
    d["len_bin"] = pd.cut(d.hl_words, [0, 4, 6, 8, 10, 100], labels=["<=4", "5-6", "7-8", "9-10", ">10"])
    t_len = d.groupby(["len_bin", "y"], observed=True).agg(n=("y", "size"), flagged_rate=("pred", "mean")).reset_index(); t_len["flagged_rate"] = (t_len.flagged_rate * 100).round(1)
    t_dom = fk.groupby("domain").agg(n_fake=("y", "size"), recall=("pred", "mean")).reset_index().sort_values("n_fake", ascending=False).head(8); t_dom["recall"] = (t_dom.recall * 100).round(1)
    fn = d[(d.y == 1) & (d.pred == 0)].sort_values("score").head(15).assign(error="missed fake (FN)"); fp = d[(d.y == 0) & (d.pred == 1)].sort_values("score", ascending=False).head(15).assign(error="false alarm (FP)")
    ex = pd.concat([fn, fp])[["error", "headline", "domain", "category", "ftype", "score"]]
    nm = f"{model}_{cond}".replace(" ", "")
    t_type.to_csv(OUT / "tables" / f"errors_{nm}_by_ftype.csv", index=False); t_cat.to_csv(OUT / "tables" / f"errors_{nm}_authentic_fpr_by_category.csv", index=False)
    t_len.to_csv(OUT / "tables" / f"errors_{nm}_by_headline_length.csv", index=False); t_dom.to_csv(OUT / "tables" / f"errors_{nm}_fake_recall_by_domain.csv", index=False)
    ex.to_csv(OUT / "tables" / f"errors_{nm}_examples.csv", index=False, encoding="utf-8-sig")
    return t_type, t_cat, t_len, t_dom

ERR_TARGETS = []
if len(SEL):
    for fam in ["classical", "transformer"]:
        r = SEL[(SEL.family == fam) & (SEL.criterion == "S2: CV F1-fake @tuned thr")]
        if len(r): ERR_TARGETS.append((r.iloc[0].model, r.iloc[0].condition))
if ("Naive Bayes", "ros") in REP and ("Naive Bayes", "ros") not in ERR_TARGETS: ERR_TARGETS.append(("Naive Bayes", "ros"))
if RUN["error_analysis"]:
    for m, c in ERR_TARGETS:
        print(f"\n===== {m} / {c} (tuned threshold) ====="); t1, t2, t3, t4 = error_tables(m, c)
        print("Fake recall by fake type:"); show(t1); print("False-positive rate of authentic articles by category:"); show(t2)

In [ ]:
# ---- shortcut baselines: what can be predicted without reading the headline's meaning? ----
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
def shortcut_baselines():
    rows = []
    yt, yv = df_tr["is_fake"].values, df_te["is_fake"].values
    def add(name, score, thr, note):
        m = full_metrics(yv, score, thr); rows.append(dict(baseline=name, note=note, precision_fake=m["precision_fake"], recall_fake=m["recall_fake"], f1_fake=m["f1_fake"], macro_f1=m["macro_f1"], mcc=m["mcc"], pr_auc=m["pr_auc"], roc_auc=m["roc_auc"], accuracy=m["accuracy"]))
    add("Majority class (flag nothing)", np.zeros(len(yv)), 0.5, "accuracy equals the authentic share")
    Xl_tr = np.c_[df_tr.hl_words, df_tr.clean_text.str.len()]; Xl_te = np.c_[df_te.hl_words, df_te.clean_text.str.len()]
    lr = make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=1000)).fit(Xl_tr, yt); add("Headline length only (words, characters)", lr.predict_proba(Xl_te)[:, 1], 0.5, "no vocabulary at all")
    oh = OneHotEncoder(handle_unknown="ignore"); D_tr = oh.fit_transform(df_tr[["domain"]]); D_te = oh.transform(df_te[["domain"]])
    lr2 = LogisticRegression(class_weight="balanced", max_iter=1000).fit(D_tr, yt); add("Publication domain only (oracle shortcut)", lr2.predict_proba(D_te)[:, 1], 0.5, "uses metadata not available to a headline model")
    add("'relation' == unrelated (metadata rule)", (df_te["relation"] == "unrelated").astype(float).values, 0.5, "headline-body mismatch flag; leaky metadata, NOT used as a feature")
    return pd.DataFrame(rows)

SHORT = shortcut_baselines()
as_pct(SHORT).round(2).to_csv(OUT / "tables" / "shortcut_baselines.csv", index=False)
print("SHORTCUT BASELINES (test set) — compare with the real models' numbers"); show(as_pct(SHORT).round(1))

In [ ]:
# ---- which n-grams drive a linear model? (training data only) ----
from sklearn.feature_selection import chi2
vec_i = make_vectorizer("word_paper"); Xi = vec_i.fit_transform(X_tr); terms = np.array(vec_i.get_feature_names_out())
lr_i = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED).fit(Xi, y_tr); coef = lr_i.coef_[0]
ch, pv_ = chi2(Xi, y_tr)
TOP = pd.concat([pd.DataFrame(dict(direction="indicates FAKE", term=terms[np.argsort(-coef)[:30]], coefficient=np.sort(coef)[::-1][:30])),
                 pd.DataFrame(dict(direction="indicates AUTHENTIC", term=terms[np.argsort(coef)[:30]], coefficient=np.sort(coef)[:30]))], ignore_index=True)
TOP["chi2"] = [ch[np.where(terms == t)[0][0]] for t in TOP.term]; TOP["chi2_p"] = [pv_[np.where(terms == t)[0][0]] for t in TOP.term]
TOP.round(4).to_csv(OUT / "tables" / "top_terms_logistic_regression.csv", index=False, encoding="utf-8-sig")
show(TOP.round(3).head(12))

## 15. Export — everything the paper needs, in one bundle

In [ ]:
def pip_freeze_subset():
    try:
        out = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout.splitlines()
        keep = [l for l in out if re.match(r"(numpy|pandas|scipy|scikit-learn|imbalanced-learn|lightgbm|matplotlib|torch|transformers|normalizer|tokenizers|joblib)[=@ ]", l, re.I)]
        return "\n".join(keep)
    except Exception as e:
        return repr(e)
(OUT / "manifest" / "pip_freeze_key_packages.txt").write_text(pip_freeze_subset(), encoding="utf-8")

KEY = OrderedDict(profile=PROFILE, seed=SEED, config_hash=CONFIG_HASH, audit=AUDIT, split=SPLIT, eda=EDA, environment=ENV,
                  bert_max_len=CONFIG["bert"]["max_len"], normalizer=globals().get("NORMALIZER"), n_configurations=int(len(AGG)))
if len(SEL): KEY["finalists"] = SEL.to_dict(orient="records")
if len(globals().get("H2H", [])): KEY["head_to_head"] = H2H.to_dict(orient="records")
if len(REPCV_SUM): KEY["repeated_cv_best_macro_f1"] = REPCV_SUM.loc[REPCV_SUM.macro_f1_mean.idxmax(), ["model", "condition", "macro_f1_mean", "f1_fake_mean", "pr_auc_mean"]].to_dict()
(OUT / "manifest" / "paper_key_numbers.json").write_text(json.dumps(KEY, indent=1, default=str), encoding="utf-8")
log("[export] key numbers written; run finished")

def make_bundle(include_cache=False):
    stamp = dt.datetime.now().strftime("%Y%m%d_%H%M")
    base = OUT_ROOT / f"DM_BanFakeNews_results_{PROFILE}_{stamp}"
    tmp = OUT_ROOT / f"_bundle_{PROFILE}"
    if tmp.exists(): shutil.rmtree(tmp)
    tmp.mkdir(parents=True)
    for sub in ["manifest", "tables", "figures", "logs"] + (["cache"] if include_cache else []): shutil.copytree(OUT / sub, tmp / sub)
    z = shutil.make_archive(str(base), "zip", tmp); shutil.rmtree(tmp); return z

BUNDLE = make_bundle(include_cache=False)
print("\nResult bundle:", BUNDLE, f"({os.path.getsize(BUNDLE) / 1e6:.1f} MB)")
print("Tables:", len(list((OUT / 'tables').glob('*.csv'))), "| Figures:", len(list((OUT / 'figures').glob('*.png'))))
if IN_COLAB:
    from google.colab import files
    files.download(BUNDLE)          # the file is also kept in your Google Drive folder